# Deteksi Multi-Label Ujaran Kebencian dan Bahasa Kasar pada Tweet Berbahasa Indonesia

Notebook ini memuat seluruh penelitian, mulai dari data dan preprocessing hingga metode, eksperimen, evaluasi, uji statistik, analisis kesalahan, eksperimen lanjutan, dan evaluasi lintas korpus. Datasetnya adalah korpus Twitter berbahasa Indonesia dari Ibrohim dan Budi (2019) dengan 12 label ujaran kebencian dan bahasa kasar.

Penelitian ini menjawab empat pertanyaan penelitian.

RQ1. Bagaimana kinerja empat keluarga algoritma, yaitu Bayesian, support vector machine, tree ensemble, dan neural network, pada 12 label jika dievaluasi dengan metrik yang sesuai untuk data tidak seimbang?

RQ2. Apakah class weighting, MLSMOTE, dan threshold tuning per label meningkatkan kinerja, terutama pada label minoritas?

RQ3. Apakah perbedaan antaralgoritma signifikan secara statistik?

RQ4. Seberapa baik model menggeneralisasi ke korpus lain, yaitu IndoToxic2024?

Bagian pertama notebook mendefinisikan semua fungsi, dikelompokkan per topik. Bagian kedua menjalankan penelitian: analisis eksploratif, preprocessing, fold, inspeksi MLSMOTE, grid model klasik, evaluasi, analisis kesalahan, eksperimen lanjutan, dan evaluasi lintas korpus. Di bagian kedua, setiap topik dipecah menjadi cell kecil: cell yang menetapkan nilai, cell yang menghitung tanpa menampilkan apa pun, dan cell yang menampilkan satu tabel atau satu grafik.

Untuk menjalankan penelitian, jalankan semua cell dari atas. Bagian yang berat dilanjutkan dari titik terakhir. Run yang sudah tercatat di results/experiments.csv atau results/experiments_ext.csv dilewati, sehingga notebook aman dijalankan ulang. XGBoost, MLP, dan transformer memakai CUDA jika tersedia.

Aturan data leakage. TF-IDF, resampling, pemilihan hiperparameter, pemilihan threshold, pemilihan epoch, dan threshold ensemble hanya memakai bagian latih dari setiap outer fold. Outer test fold tidak pernah dipakai untuk memilih apa pun.

# Glosarium

Kode singkat yang disimpan di file hasil, beserta istilah baku yang dipakai di notebook ini dan di paper.

| Kelompok | Kode | Arti |
|---|---|---|
| Algoritma | cnb | Complement Naive Bayes |
| Algoritma | svm | Support vector machine linear (LinearSVC) |
| Algoritma | xgb | XGBoost |
| Algoritma | mlp | Multilayer perceptron |
| Algoritma | rf | Random forest, hanya untuk replikasi baseline Label Powerset |
| Transformer | tf | Transformer yang di-fine-tune dengan protokol revisi |
| Transformer | ens | Ensemble yang merata-ratakan probabilitas beberapa model tf |
| Transformer | bert | Baris dari protokol IndoBERTweet sebelumnya. Protokol ini berhenti di batas epoch pada setiap fold, sehingga barisnya tetap ada di file tetapi dikeluarkan dari semua analisis |
| Strategi multi-label | br | Binary Relevance |
| Strategi multi-label | cc | Classifier Chains |
| Strategi multi-label | lp | Label Powerset |
| Strategi multi-label | mo | Satu output layer multi-label, yaitu cara transformer memprediksi semua label sekaligus |
| Penanganan ketidakseimbangan | none | Tanpa penanganan |
| Penanganan ketidakseimbangan | weight | Class weighting |
| Penanganan ketidakseimbangan | mlsmote | MLSMOTE dengan hierarchy repair |
| Penanganan ketidakseimbangan | mlsmote_raw | MLSMOTE asli tanpa repair |
| Penanganan ketidakseimbangan | mlros | ML-ROS |
| Mode tuning | fixed | Hiperparameter tetap dan threshold default |
| Mode tuning | thr | Hiperparameter tetap dengan satu threshold per label yang dipilih pada inner validation |
| Mode tuning | tuned | Pencarian hiperparameter kecil ditambah satu threshold per label, keduanya dipilih pada inner validation |
| Fitur | word | TF-IDF unigram dan bigram kata |
| Fitur | char | TF-IDF 3 sampai 5 gram karakter |
| Fitur | wordchar | Gabungan word dan char |
| Metrik | jaccard_acc | Indeks Jaccard atau example-based accuracy, sama dengan Persamaan 1 pada Ibrohim dan Budi (2019) |
| Metrik | subset_acc | Subset accuracy, disebut juga exact match ratio |
| Metrik | minority_macro_f1 | Macro F1 atas lima label minoritas |
| Metrik | pred_hierarchy_violation_rate | Proporsi label set prediksi yang melanggar hierarki anotasi |
| Uji statistik | delta_macro_f1 | Selisih macro F1, yaitu model B dikurangi model A |
| Uji statistik | ci95_low, ci95_high | Batas interval kepercayaan 95 persen untuk selisih tersebut |
| Uji statistik | p_permutation | p-value dari paired permutation test |
| Uji statistik | p_corrected_t | p-value dari corrected resampled t-test |
| Uji statistik | p_equivalence | p-value dari equivalence test terhadap margin |
| Uji statistik | akhiran _holm | p-value setelah koreksi Holm-Bonferroni |
| Uji statistik | verdict | Hasil perbandingan: B better, A better, equivalent, atau inconclusive |
| Nested evaluation | outer fold | Fold yang hanya dipakai untuk pengujian |
| Nested evaluation | inner validation | 15 persen dari setiap outer training fold yang disisihkan untuk tuning |

Kolom condition di file hasil ditampilkan sebagai imbalance_treatment di tabel notebook. Perbandingan fixed dengan thr memisahkan efek threshold tuning, dan perbandingan thr dengan tuned memisahkan efek pencarian hiperparameter.

ID konfigurasi disusun dari kode di atas, misalnya svm__br__none__thr atau svm__br__none__tuned. Run transformer memakai awalan tf dan ens. ID tidak diubah agar run yang tersimpan dapat dipakai ulang.

# Konfigurasi

Semua switch yang menentukan bagian penelitian mana yang dijalankan ada di cell di bawah. Switch yang bernilai None ditentukan otomatis dari hardware yang tersedia.

In [ ]:
# Konfigurasi. Ubah nilai hanya di cell ini.
RANDOM_STATE = 42
RESULTS_CSV = "experiments.csv"  # log run grid utama, satu baris per konfigurasi dan fold
EXTENDED_CSV = "experiments_ext.csv"  # log run eksperimen lanjutan
RUN_GRID = True  # grid klasik, dilanjutkan otomatis
RUN_GPU_MODELS = None  # XGBoost dan MLP, None berarti dijalankan jika CUDA tersedia
RUN_EXTENDED_GRID = True  # grid ML-ROS dan n-gram karakter, dilanjutkan otomatis
RUN_TRANSFORMERS = None  # protokol transformer dan ensemble, None berarti otomatis
RUN_CROSS_CORPUS = None  # None berarti dijalankan jika file IndoToxic2024 tersedia
RUN_INFERENCE_BENCHMARK = True  # ukur waktu prediksi 1.000 tweet tiap model pembanding
REMOVE_STALE_RUNS = True  # True menghapus run dari grid tuning lama agar dijalankan ulang
REMOVE_OTHER_DEVICE_RUNS = True  # True menjalankan ulang run XGBoost dan MLP dari GPU lain
THRESHOLD_ONLY_ALGOS = ["cnb", "svm"]  # algoritma pada ablation threshold saja
MLROS_ALGOS = ["cnb", "svm"]  # algoritma pada eksperimen ML-ROS
CHAR_FEATURE_ALGOS = ["svm"]  # algoritma pada eksperimen n-gram karakter
EQUIVALENCE_MARGIN = 0.01  # selisih macro F1 terkecil yang dianggap bermakna
N_PERMUTATIONS = 5000  # jumlah permutasi untuk paired permutation test
N_BOOTSTRAP = 2000  # sampel bootstrap untuk interval kepercayaan lintas korpus
N_JOBS_CPU = None  # None berarti jumlah CPU dikurangi satu, paling banyak empat
DATA_REPO = (
    "https://github.com/okkyibrohim/id-multi-label-hate-speech-and-abusive-language-detection"
)

# Persiapan

Bagian ini mengimpor library, membuat folder, mengunduh dataset, menetapkan random seed, dan mendeteksi device komputasi. PyTorch dan transformers bersifat opsional. Tanpa keduanya, bagian MLP dan transformer dilewati dan hasil tersimpannya yang dipakai.

## Impor Library

Semua library yang dipakai di notebook diimpor di sini. Di Google Colab, paket yang belum tersedia dipasang terlebih dahulu.

In [ ]:
import ast
import json
import math
import os
import random
import re
import shutil
import subprocess
import sys
import time
from itertools import combinations, product
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "iterative-stratification",
            "xgboost>=2.0",
            "emoji>=2.10",
        ],
        check=True,
    )

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse as sp
import seaborn as sns
from IPython.display import display
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold, MultilabelStratifiedShuffleSplit
from joblib import Parallel, delayed
from scipy import stats
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    hamming_loss,
    jaccard_score,
    precision_recall_fscore_support,
    precision_score,
    recall_score,
)
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.naive_bayes import ComplementNB
from sklearn.svm import LinearSVC
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

try:
    import torch
    from torch import nn
except ImportError:  # PyTorch dipasang terpisah, lihat requirements.txt
    torch = nn = None
try:
    from transformers import (
        AutoModelForSequenceClassification,
        AutoTokenizer,
        get_linear_schedule_with_warmup,
    )
except ImportError:
    AutoModelForSequenceClassification = AutoTokenizer = get_linear_schedule_with_warmup = None
try:
    import emoji
except ImportError:
    emoji = None

## Siapkan Folder dan Dataset

Folder untuk data, hasil, dan gambar dibuat, dan dataset di-clone dari repositori penulis dataset pada run pertama.

In [ ]:
ROOT = Path.cwd()
RAW, PROC, EXTERNAL = ROOT / "data" / "raw", ROOT / "data" / "processed", ROOT / "data" / "external"
RES, FIG = ROOT / "results", ROOT / "figures"
FOLDS_PATH = RES / "folds.json"
for folder in [RAW, PROC, RES, FIG, RES / "preds", RES / "preds_ext"]:
    folder.mkdir(parents=True, exist_ok=True)

if not (RAW / "re_dataset.csv").exists():  # unduh dataset pada run pertama
    source_dir = ROOT / "data" / "_src"
    if not source_dir.exists():
        subprocess.run(
            ["git", "clone", "-q", "--depth", "1", DATA_REPO, str(source_dir)], check=True
        )
    for name in ["re_dataset.csv", "new_kamusalay.csv", "abusive.csv", "citation.bib"]:
        (RAW / name).write_bytes((source_dir / name).read_bytes())

## Tetapkan Seed dan Device

Random seed dan opsi tampilan ditetapkan, lalu device komputasi dideteksi. Di GPU, cuDNN berjalan dalam mode deterministik agar MLP memberikan hasil yang sama di setiap run. Switch yang dibiarkan None di konfigurasi ditentukan dari device.

In [ ]:
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 200)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "savefig.bbox": "tight"})

DEVICE = "cuda" if torch is not None and torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    GPU_NAME = torch.cuda.get_device_name(0)
    torch.backends.cudnn.deterministic = True  # pelatihan MLP yang dapat direproduksi
    torch.backends.cudnn.benchmark = False
else:
    GPU_NAME = "-" if torch is not None else "- (PyTorch not installed)"
DEVICE_LABEL = f"cuda ({GPU_NAME})" if DEVICE == "cuda" else "cpu"
if RUN_GPU_MODELS is None:
    RUN_GPU_MODELS = DEVICE == "cuda"
if RUN_TRANSFORMERS is None:
    RUN_TRANSFORMERS = DEVICE == "cuda"
if N_JOBS_CPU is None:
    N_JOBS_CPU = max(1, min(4, (os.cpu_count() or 2) - 1))
pd.DataFrame(
    [
        {
            "root": str(ROOT),
            "device": DEVICE,
            "gpu": GPU_NAME,
            "gpu_models": RUN_GPU_MODELS,
            "transformers": RUN_TRANSFORMERS,
            "cpu_jobs": N_JOBS_CPU,
        }
    ]
)

# Label dan Hierarki Anotasi

Ke-12 label mengikuti hierarki yang diperiksa terhadap data pada analisis eksploratif. Jika HS negatif, semua child label HS negatif. Jika HS positif, sebuah tweet memiliki tepat satu target, tepat satu level, dan minimal satu kategori. HS_Other tidak pernah muncul bersama kategori spesifik, dan Abusive tidak bergantung pada HS.

Aturan ini dipakai di tiga tempat: untuk memvalidasi data, untuk memperbaiki synthetic label set di dalam MLSMOTE, dan untuk post-processing prediksi pada eksperimen lanjutan.

## Kelompok Label

Ke-12 label dan kelompoknya. Lima label minoritas adalah label yang imbalance ratio-nya, IRLbl, berada di atas mean imbalance ratio, MeanIR.

In [ ]:
LABELS = [
    "HS",
    "Abusive",
    "HS_Individual",
    "HS_Group",
    "HS_Religion",
    "HS_Race",
    "HS_Physical",
    "HS_Gender",
    "HS_Other",
    "HS_Weak",
    "HS_Moderate",
    "HS_Strong",
]
LABEL_INDEX = {name: index for index, name in enumerate(LABELS)}
TARGET_LABELS = ["HS_Individual", "HS_Group"]
SPECIFIC_CATEGORIES = ["HS_Religion", "HS_Race", "HS_Physical", "HS_Gender"]
OTHER_CATEGORY = "HS_Other"
LEVEL_LABELS = ["HS_Weak", "HS_Moderate", "HS_Strong"]
HS_CHILDREN = TARGET_LABELS + SPECIFIC_CATEGORIES + [OTHER_CATEGORY] + LEVEL_LABELS
CATEGORY_LABELS = SPECIFIC_CATEGORIES + [OTHER_CATEGORY]
MINORITY_LABELS = ["HS_Religion", "HS_Race", "HS_Physical", "HS_Gender", "HS_Strong"]

## Indeks Label

Mengubah nama label menjadi posisi kolom. Posisi setiap kelompok label dihitung sekali.

In [ ]:
def label_indices(names):
    return [LABEL_INDEX[name] for name in names]


HS_INDEX = LABEL_INDEX["HS"]
CHILD_INDICES = label_indices(HS_CHILDREN)
TARGET_INDICES = label_indices(TARGET_LABELS)
LEVEL_INDICES = label_indices(LEVEL_LABELS)
SPECIFIC_INDICES = label_indices(SPECIFIC_CATEGORIES)
OTHER_INDEX = LABEL_INDEX[OTHER_CATEGORY]
MINORITY_INDICES = label_indices(MINORITY_LABELS)

## Mask Aturan Hierarki

Mengembalikan satu mask boolean per aturan anotasi. Sebuah baris bernilai true jika melanggar aturan tersebut.

In [ ]:
def hierarchy_rule_masks(Y):
    Y = np.asarray(Y).astype(int)
    hate_speech = Y[:, HS_INDEX]
    n_children = Y[:, CHILD_INDICES].sum(axis=1)
    n_targets = Y[:, TARGET_INDICES].sum(axis=1)
    n_levels = Y[:, LEVEL_INDICES].sum(axis=1)
    n_specific = Y[:, SPECIFIC_INDICES].sum(axis=1)
    other = Y[:, OTHER_INDEX]
    is_hs = hate_speech == 1
    return {
        "hs0_but_child1": (hate_speech == 0) & (n_children > 0),
        "hs1_target_not_exactly_one": is_hs & (n_targets != 1),
        "hs1_level_not_exactly_one": is_hs & (n_levels != 1),
        "hs1_no_category": is_hs & (n_specific + other == 0),
        "other_with_specific": (other == 1) & (n_specific > 0),
    }

## Mask Pelanggaran Hierarki

Menandai baris yang melanggar minimal satu aturan anotasi.

In [ ]:
def hierarchy_violation_mask(Y):
    return np.logical_or.reduce(list(hierarchy_rule_masks(Y).values()))

## Hitung Pelanggaran Hierarki

Menghitung baris yang melanggar setiap aturan dan baris yang melanggar aturan mana pun.

In [ ]:
def count_hierarchy_violations(Y):
    masks = hierarchy_rule_masks(Y)
    counts = {rule: int(mask.sum()) for rule, mask in masks.items()}
    counts["any_violation_rows"] = int(np.logical_or.reduce(list(masks.values())).sum())
    return counts

## Pertahankan Label Tertinggi dalam Kelompok

Menjadikan label dengan skor tertinggi dalam satu kelompok positif dan sisa kelompok negatif. Baris diubah secara in-place.

In [ ]:
def keep_highest(y_row, scores, group):
    best = group[int(np.argmax(scores[group]))]
    y_row[group] = 0
    y_row[best] = 1

## Selesaikan Kategori

Memastikan baris ujaran kebencian memiliki minimal satu kategori. Jika HS_Other muncul bersama kategori spesifik, sisi dengan skor lebih tinggi yang dipertahankan. Baris diubah secara in-place.

In [ ]:
def resolve_categories(y_row, scores):
    has_specific = y_row[SPECIFIC_INDICES].sum() > 0
    has_other = y_row[OTHER_INDEX] == 1
    if has_specific and has_other:
        if scores[OTHER_INDEX] > scores[SPECIFIC_INDICES].max():
            y_row[SPECIFIC_INDICES] = 0
        else:
            y_row[OTHER_INDEX] = 0
    elif not has_specific and not has_other:
        categories = SPECIFIC_INDICES + [OTHER_INDEX]
        y_row[categories[int(np.argmax(scores[categories]))]] = 1

## Repair Synthetic Label Set

Membuat satu synthetic label set dari MLSMOTE mengikuti hierarki. Skornya adalah frekuensi label di antara seed dan tetangganya, dan label set milik seed memecah skor yang sama, sehingga repair mengikuti seed.

In [ ]:
def repair_synthetic_labelset(labelset, scores, seed_y):
    labelset = np.asarray(labelset).astype(int).copy()
    priority = np.asarray(scores, dtype=float) + 0.5 * np.asarray(seed_y).astype(
        int
    )  # seed menang jika skornya sama
    if labelset[CHILD_INDICES].sum() > 0:
        labelset[HS_INDEX] = 1
    if labelset[HS_INDEX] == 0:
        labelset[CHILD_INDICES] = 0
        return labelset
    keep_highest(labelset, priority, TARGET_INDICES)
    keep_highest(labelset, priority, LEVEL_INDICES)
    resolve_categories(labelset, priority)
    return labelset

## Terapkan Hierarki Label

Mengubah decision score menjadi label set yang mengikuti hierarki. Skor positif berarti label berada di atas threshold-nya. Pada mode topdown, HS ditentukan hanya oleh skor HS, dan HS negatif mengosongkan semua child label HS. Pada mode bottomup, HS menjadi positif jika skor HS atau skor salah satu child label-nya positif. Aturan ini berasal dari pedoman anotasi dataset, bukan dari data uji.

In [ ]:
def enforce_label_hierarchy(scores, mode="topdown"):
    scores = np.asarray(scores, dtype=float)
    Y = (scores > 0).astype(int)
    hate_speech = Y[:, HS_INDEX].astype(bool)
    if mode == "bottomup":
        hate_speech = hate_speech | (Y[:, CHILD_INDICES].sum(1) > 0)
    elif mode != "topdown":
        raise ValueError(mode)
    Y[:, HS_INDEX] = hate_speech.astype(int)
    Y[np.ix_(~hate_speech, CHILD_INDICES)] = 0
    for row in np.where(hate_speech)[0]:
        keep_highest(Y[row], scores[row], TARGET_INDICES)
        keep_highest(Y[row], scores[row], LEVEL_INDICES)
        resolve_categories(Y[row], scores[row])
    return Y

# Preprocessing Teks

Model klasik memakai teks yang sudah dibersihkan dan dinormalisasi slang-nya. Tidak ada stemming dan tidak ada penghapusan stopword, mengikuti Hana et al. (2020) dan Hendrawan et al. (2020). Transformer menerima tweet asli dengan normalisasi ringan yang dipakai IndoBERTweet (Koto et al., 2021).

## Muat Dataset Mentah

Memuat re_dataset.csv. File ini memakai encoding latin-1, sehingga decoding UTF-8 gagal dan encoding kedua yang dipakai.

In [ ]:
def load_raw_dataset(path):
    try:
        table = pd.read_csv(path, encoding="utf-8")
    except UnicodeDecodeError:
        table = pd.read_csv(path, encoding="latin-1")
    missing = [column for column in ["Tweet"] + LABELS if column not in table.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")
    return table

## Muat Kamus Slang

Memuat new_kamusalay.csv, yang tidak memiliki header dan berisi satu kata slang beserta bentuk bakunya di setiap baris.

In [ ]:
def load_slang_dictionary(path):
    table = pd.read_csv(path, encoding="latin-1", header=None, names=["slang", "standard"])
    table = table.dropna().drop_duplicates("slang", keep="first")
    return dict(zip(table["slang"].str.strip(), table["standard"].str.strip()))

## Muat Leksikon Kata Kasar

Memuat daftar kata kasar yang disertakan bersama dataset. Daftar ini hanya dipakai untuk menyamarkan kata kasar pada contoh yang ditampilkan.

In [ ]:
def load_abusive_lexicon(path):
    return set(pd.read_csv(path, encoding="latin-1")["ABUSIVE"].str.strip().str.lower())

## Hapus Emoji Ter-escape dan Baris Baru

Dataset menyimpan emoji sebagai teks byte ter-escape dan baris baru sebagai teks ter-escape. Keduanya diganti dengan spasi.

In [ ]:
_RE_ESCAPED_BYTES = re.compile(r"\\x[0-9a-fA-F]{2}")
_RE_ESCAPED_NEWLINE = re.compile(r"\\n")


def remove_escaped_bytes(text):
    return _RE_ESCAPED_NEWLINE.sub(" ", _RE_ESCAPED_BYTES.sub(" ", text))

## Hapus URL dan Entitas HTML

Alamat web dan entitas HTML, seperti ampersand yang di-encode, diganti dengan spasi.

In [ ]:
_RE_HTTP = re.compile(r"https?://\S+|www\.\S+")
_RE_HTML_ENTITY = re.compile(r"&[a-z]+;")


def remove_urls_and_entities(text):
    return _RE_HTML_ENTITY.sub(" ", _RE_HTTP.sub(" ", text))

## Hapus Token Twitter

Dataset mengganti nama pengguna dengan USER dan tautan dengan URL, serta mempertahankan RT untuk retweet. Token ini tidak bermakna sehingga dihapus.

In [ ]:
_RE_SPECIAL_TOKENS = re.compile(r"\b(?:user|url|rt)\b")


def remove_twitter_tokens(text):
    return _RE_SPECIAL_TOKENS.sub(" ", text)

## Uraikan Kata Ulang

Bahasa Indonesia menulis kata ulang dengan angka 2, sehingga kita2 menjadi kita kita dan lucu2an menjadi lucu lucuan.

In [ ]:
_RE_REDUPLICATION = re.compile(r"\b([a-z]{2,})2(?![0-9])")


def expand_reduplication(text):
    return _RE_REDUPLICATION.sub(r"\1 \1", text)

## Hapus Angka dan Tanda Baca

Angka dihapus tanpa memecah kata, sehingga g30spki menjadi gspki. Tanda baca, simbol, dan sisa karakter non ASCII kemudian diganti dengan spasi.

In [ ]:
_RE_DIGIT = re.compile(r"[0-9]+")
_RE_NON_ALPHA = re.compile(r"[^a-z\s]")


def remove_digits_and_punctuation(text):
    return _RE_NON_ALPHA.sub(" ", _RE_DIGIT.sub("", text))

## Ringkas Huruf Berulang

Huruf yang berulang tiga kali atau lebih diringkas menjadi satu, sehingga bangeeett menjadi banget.

In [ ]:
_RE_REPEATED_CHAR = re.compile(r"([a-z])\1{2,}")


def collapse_repeated_letters(text):
    return _RE_REPEATED_CHAR.sub(r"\1", text)

## Bersihkan Teks

Menjalankan langkah pembersihan di atas secara berurutan setelah teks diubah ke huruf kecil, lalu menghapus spasi berlebih.

In [ ]:
_RE_SPACES = re.compile(r"\s+")


def clean_text(text):
    cleaned = remove_escaped_bytes(str(text)).lower()
    cleaned = remove_twitter_tokens(remove_urls_and_entities(cleaned))
    cleaned = remove_digits_and_punctuation(expand_reduplication(cleaned))
    return _RE_SPACES.sub(" ", collapse_repeated_letters(cleaned)).strip()

## Normalisasi Slang

Mengganti setiap token slang dengan bentuk bakunya. Satu penggantian bisa terdiri atas lebih dari satu kata.

In [ ]:
def normalize_slang(text, slang_dict):
    return " ".join(slang_dict.get(token, token) for token in text.split())

## Bangun Fungsi Preprocessing

Mengembalikan fungsi preprocessing untuk model klasik: pembersihan yang diikuti normalisasi slang.

In [ ]:
def build_preprocessor(slang_dict):
    def preprocess(text):
        return normalize_slang(clean_text(text), slang_dict)

    return preprocess

## Normalisasi Teks untuk Transformer

Normalisasi ringan dengan gaya IndoBERTweet. Byte ter-escape di-decode agar emoji menjadi karakter asli, emoji diubah menjadi kata, USER menjadi @USER, dan URL menjadi HTTPURL.

In [ ]:
_RE_ESCAPED_BYTE_RUN = re.compile(r"(?:\\x[0-9a-fA-F]{2})+")


def decode_escaped_bytes(match):
    raw = bytes(
        int(hex_pair, 16) for hex_pair in re.findall(r"\\x([0-9a-fA-F]{2})", match.group(0))
    )
    return " " + raw.decode("utf-8", errors="ignore") + " "


def normalize_for_transformer(text):
    normalized = re.sub(r"^b['\"]", "", str(text)).rstrip("'\"")
    normalized = _RE_ESCAPED_BYTE_RUN.sub(decode_escaped_bytes, normalized.replace("\\n", " "))
    if emoji is not None:
        normalized = emoji.demojize(normalized, delimiters=(" ", " "))
    normalized = re.sub(r"\bRT\b", " ", normalized)
    normalized = re.sub(r"\bUSER\b", "@USER", normalized)
    normalized = re.sub(r"\bURL\b", "HTTPURL", normalized)
    return re.sub(r"\s+", " ", normalized).strip()

## Hapus Duplikat

Duplikat teks bersih dengan label identik disisakan satu salinan. Duplikat dengan label berbeda adalah konflik anotasi dan semuanya dihapus, karena tidak ada dasar untuk memilih label set yang benar. Baris yang teksnya kosong setelah dibersihkan dihapus. Duplikat yang tersebar di training fold dan test fold akan menimbulkan data leakage, karena model dapat menghafal teksnya.

In [ ]:
def deduplicate(table, text_col="text_clean"):
    n_raw = len(table)
    empty = table[text_col].str.len() == 0
    table = table[~empty]
    n_label_variants = table.groupby(text_col)[LABELS].nunique().max(axis=1)
    conflict_texts = set(n_label_variants[n_label_variants > 1].index)
    is_conflict = table[text_col].isin(conflict_texts)
    df_ok = table[~is_conflict]
    same_label_dup = df_ok.duplicated(text_col, keep="first")
    df_out = df_ok[~same_label_dup].reset_index(drop=True)
    report = {
        "rows_raw": n_raw,
        "rows_empty_after_clean": int(empty.sum()),
        "conflicting_texts": len(conflict_texts),
        "rows_dropped_conflict": int(is_conflict.sum()),
        "rows_dropped_same_label_dup": int(same_label_dup.sum()),
        "rows_final": len(df_out),
    }
    return df_out, report

## Samarkan Kata Kasar

Menyamarkan kata kasar untuk ditampilkan, dengan hanya mempertahankan huruf pertama dan terakhirnya.

In [ ]:
def mask_abusive_words(text, abusive_lexicon):
    out = []
    for word in str(text).split():
        core = re.sub(r"[^a-zA-Z]", "", word).lower()
        if core in abusive_lexicon and len(core) > 2:
            out.append(word[0] + "*" * (len(word) - 2) + word[-1])
        else:
            out.append(word)
    return " ".join(out)

## Matriks Label

Mengembalikan 12 label dari sebuah data frame sebagai matriks bilangan bulat.

In [ ]:
def label_matrix(table):
    return table[LABELS].to_numpy(dtype=int)

# Fitur

TfidfVectorizer scikit-learn memilih vocabulary-nya dengan sorting yang tidak stabil. Dataset ini memiliki ratusan term dengan frekuensi sama di batas cut-off. Pada fold 0, 921 term berfrekuensi 6 dan hanya 202 di antaranya yang dipertahankan, sehingga hasil dapat berbeda antarmesin hingga 0,014 macro F1. TF-IDF deterministik di bawah memakai kriteria yang sama, yaitu frekuensi term pada teks latih, dan memecah frekuensi yang sama secara alfabetis, sehingga vocabulary-nya dapat direproduksi.

## TF-IDF Deterministik

Mempertahankan term latih yang paling sering muncul, memecah frekuensi yang sama secara alfabetis, dan melatih TF-IDF dengan vocabulary tetap tersebut.

In [ ]:
def fit_deterministic_tfidf(texts, ngram_range, max_features, sublinear_tf, analyzer, min_df=1):
    counter = CountVectorizer(ngram_range=ngram_range, min_df=min_df, analyzer=analyzer)
    counts = counter.fit_transform(texts)
    terms = counter.get_feature_names_out()
    term_freq = np.asarray(counts.sum(axis=0)).ravel()
    order = np.lexsort(
        (terms, -term_freq)
    )  # frekuensi tertinggi lebih dulu, nilai sama diurutkan alfabetis
    keep = order[:max_features] if max_features else order
    vocabulary = sorted(terms[keep].tolist())
    return TfidfVectorizer(
        ngram_range=ngram_range, sublinear_tf=sublinear_tf, analyzer=analyzer, vocabulary=vocabulary
    ).fit(texts)

## Feature Set

Fitur kata adalah unigram dan bigram dengan 5.000 term dan sublinear term frequency. Fitur karakter adalah 3 sampai 5 gram karakter di dalam batas kata dengan 20.000 term. Feature set wordchar menumpuk keduanya. Fungsi ini melatih satu TF-IDF untuk setiap bagian dari feature set yang dipilih pada teks latih.

In [ ]:
WORD_TFIDF_PARAMS = {
    "ngram_range": (1, 2),
    "max_features": 5000,
    "sublinear_tf": True,
    "analyzer": "word",
}
CHAR_TFIDF_PARAMS = {
    "ngram_range": (3, 5),
    "max_features": 20000,
    "sublinear_tf": True,
    "analyzer": "char_wb",
}
FEATURE_SETS = {
    "word": [WORD_TFIDF_PARAMS],
    "char": [CHAR_TFIDF_PARAMS],
    "wordchar": [WORD_TFIDF_PARAMS, CHAR_TFIDF_PARAMS],
}


def fit_features(texts, features="word"):
    return [fit_deterministic_tfidf(texts, **params) for params in FEATURE_SETS[features]]

## Transformasi Fitur

Mengubah teks menjadi matriks fitur dengan model TF-IDF yang sudah dilatih, dan menumpuk bagian-bagiannya berdampingan jika lebih dari satu.

In [ ]:
def transform_features(vectorizers, texts):
    matrices = [vectorizer.transform(texts) for vectorizer in vectorizers]
    return matrices[0] if len(matrices) == 1 else sp.hstack(matrices, format="csr")

# Resampling

MLSMOTE (Charte et al., 2015, Knowledge-Based Systems) mengikuti Algoritma 1 dan 2 dari paper aslinya. Label minoritas adalah label yang IRLbl-nya di atas MeanIR, yang dihitung sekali. Tetangga dicari di antara instance setiap label minoritas, fitur sintetis menginterpolasi seed dan satu tetangga acak, dan synthetic label set mengikuti aturan ranking dari paper. Dua tambahannya adalah hierarchy repair yang bersifat opsional dan pencarian tetangga yang deterministik.

ML-ROS (Charte et al., 2015, Neurocomputing) menggandakan instance acak dari setiap label minoritas secara bergiliran hingga jumlah salinan mencapai 20 persen dari data latih. Angka ini mendekati proporsi yang ditambahkan MLSMOTE, yaitu rata-rata 20,6 persen.

## Imbalance Ratio per Label

IRLbl membagi jumlah label yang paling sering muncul dengan jumlah setiap label. Label tanpa instance positif bernilai tak hingga.

In [ ]:
def irlbl(Y):
    counts = np.asarray(Y).sum(axis=0).astype(float)
    with np.errstate(divide="ignore"):
        return np.where(counts > 0, counts.max() / counts, np.inf)

## Mean Imbalance Ratio

MeanIR adalah rata-rata dari nilai IRLbl yang terhingga.

In [ ]:
def mean_ir(Y):
    ratios = irlbl(Y)
    return float(ratios[np.isfinite(ratios)].mean())

## Tetangga Terdekat

Mencari tetangga terdekat setiap instance berdasarkan jarak Euclidean. Jarak dibulatkan untuk menghilangkan perbedaan floating point pada digit terakhir, dan jarak yang sama dipecah berdasarkan indeks, sehingga hasilnya sama di setiap mesin.

In [ ]:
def nearest_neighbours(X_bag, n_neighbours):
    distances = np.round(euclidean_distances(X_bag), 9)
    np.fill_diagonal(distances, np.inf)
    return np.argsort(distances, axis=1, kind="stable")[:, :n_neighbours]

## Synthetic Label Set

Sebuah label diberikan kepada synthetic instance jika label itu muncul pada lebih dari separuh gabungan seed dan tetangganya, sesuai rumus Algoritma 2 pada paper. Dengan repair, label set tersebut kemudian dibuat mengikuti hierarki.

In [ ]:
def synthetic_labelset(Y_bag, seed_index, neighbours, repair):
    counts = Y_bag[seed_index].astype(int) + Y_bag[neighbours].sum(axis=0)
    y_new = (counts > (len(neighbours) + 1) / 2.0).astype(np.int8)
    if not repair:
        return y_new, False
    repaired = np.asarray(repair_synthetic_labelset(y_new, counts, Y_bag[seed_index])).astype(
        np.int8
    )
    return repaired, not np.array_equal(repaired, y_new)

## MLSMOTE untuk Satu Label

Membuat satu synthetic instance untuk setiap instance dari satu label minoritas. Fiturnya terletak pada garis antara seed dan satu tetangga acak.

In [ ]:
def mlsmote_for_label(X_bag, Y_bag, n_neighbours, repair, rng):
    neighbours = nearest_neighbours(X_bag, min(n_neighbours, X_bag.shape[0] - 1))
    new_X, new_Y, n_repaired = [], [], 0
    for seed_index in range(X_bag.shape[0]):
        neigh = list(neighbours[seed_index])
        ref = neigh[rng.integers(len(neigh))]
        gap = rng.random()
        new_X.append(X_bag[seed_index] + gap * (X_bag[ref] - X_bag[seed_index]))
        y_new, was_repaired = synthetic_labelset(Y_bag, seed_index, neigh, repair)
        new_Y.append(y_new)
        n_repaired += was_repaired
    return new_X, new_Y, n_repaired

## MLSMOTE

Menelusuri label secara berurutan dan melakukan oversampling pada setiap label yang masih minoritas. IRLbl diperbarui setelah setiap label, sedangkan MeanIR tetap pada nilai awalnya. Mengembalikan data hasil resampling dan ringkasannya.

In [ ]:
def mlsmote(X, Y, n_neighbours=5, repair=True, random_state=RANDOM_STATE):
    rng = np.random.default_rng(random_state)
    is_sparse = sp.issparse(X)
    stack = (lambda parts: sp.vstack(parts, format="csr")) if is_sparse else np.vstack
    X = sp.csr_matrix(X, dtype=np.float64) if is_sparse else np.asarray(X, dtype=np.float64)
    Y = np.asarray(Y).astype(np.int8)
    m_ir = mean_ir(Y)  # dihitung sekali, seperti pada Algoritma 1
    X_parts, Y_parts = [X], [Y]
    counts = Y.sum(axis=0).astype(float)
    summary = {
        "mean_ir": m_ir,
        "irlbl_before": irlbl(Y).tolist(),
        "processed_labels": [],
        "n_synthetic": 0,
        "n_repaired": 0,
        "n_skipped_small_bag": 0,
    }
    for label in range(Y.shape[1]):
        with np.errstate(divide="ignore"):
            ir_now = counts.max() / counts[label] if counts[label] > 0 else np.inf
        if not (ir_now > m_ir) or not np.isfinite(ir_now):
            continue
        Y_all = np.vstack(Y_parts)
        bag = np.where(Y_all[:, label] == 1)[0]
        if len(bag) < 2:
            summary["n_skipped_small_bag"] += 1
            continue
        new_X, new_Y, n_repaired = mlsmote_for_label(
            stack(X_parts)[bag], Y_all[bag], n_neighbours, repair, rng
        )
        X_parts.append(stack(new_X))
        Y_parts.append(np.vstack(new_Y).astype(np.int8))
        counts += Y_parts[-1].sum(axis=0)
        summary["processed_labels"].append(int(label))
        summary["n_synthetic"] += len(new_Y)
        summary["n_repaired"] += n_repaired
    Y_out = np.vstack(Y_parts).astype(int)
    summary["irlbl_after"] = irlbl(Y_out).tolist()
    summary["mean_ir_after"] = mean_ir(Y_out)
    return stack(X_parts), Y_out, summary

## ML-ROS

Menggandakan instance acak dari setiap label minoritas secara bergiliran. Sebuah label keluar dari giliran setelah IRLbl-nya tidak lagi di atas MeanIR. Mengembalikan data hasil resampling dan ringkasannya.

In [ ]:
MLROS_PERCENTAGE = 0.20  # jumlah salinan sebagai proporsi data latih


def mlros(X, Y, percentage=MLROS_PERCENTAGE, random_state=RANDOM_STATE):
    rng = np.random.default_rng(random_state)
    Y = np.asarray(Y).astype(int)
    m_ir = mean_ir(Y)
    counts = Y.sum(0).astype(float)
    ir_start = irlbl(Y)
    is_minority = np.isfinite(ir_start) & (ir_start > m_ir)
    bags = {
        label_index: np.where(Y[:, label_index] == 1)[0]
        for label_index in range(Y.shape[1])
        if is_minority[label_index]
    }
    n_clones = int(round(percentage * len(Y)))
    picked = []
    while len(picked) < n_clones and bags:
        for label_index in list(bags):
            if len(picked) >= n_clones:
                break
            row_index = int(bags[label_index][rng.integers(len(bags[label_index]))])
            picked.append(row_index)
            counts += Y[row_index]
            if counts.max() / counts[label_index] <= m_ir:
                del bags[label_index]
    picked = np.asarray(picked, dtype=int)
    summary = {
        "n_synthetic": len(picked),
        "n_repaired": 0,
        "mean_ir": m_ir,
        "minority_labels": [LABELS[label_index] for label_index in np.where(is_minority)[0]],
    }
    if len(picked) == 0:
        return X, Y, summary
    X_out = sp.vstack([X, X[picked]], format="csr") if sp.issparse(X) else np.vstack([X, X[picked]])
    return X_out, np.vstack([Y, Y[picked]]), summary

## Resampling Data Latih

Menerapkan resampling dari sebuah penanganan ketidakseimbangan pada data latih. mlsmote memakai hierarchy repair, mlsmote_raw adalah algoritma aslinya, dan mlros adalah ML-ROS.

In [ ]:
RESAMPLING_CONDITIONS = {"mlsmote", "mlsmote_raw", "mlros"}


def resample(X, Y, condition):
    if condition == "mlros":
        return mlros(X, Y)
    return mlsmote(X, Y, n_neighbours=5, repair=(condition == "mlsmote"))

# Classifier Dasar

Model Bayesian adalah Complement Naive Bayes, support vector machine adalah LinearSVC, tree ensemble adalah XGBoost, dan neural network adalah MLP yang ditulis dengan PyTorch. XGBoost dan MLP memakai CUDA jika tersedia. Random forest hanya dipakai untuk mereplikasi baseline Label Powerset.

Class weighting berarti balanced sample weight untuk Complement Naive Bayes, balanced class weight untuk LinearSVC, scale_pos_weight untuk XGBoost, dan positive class weight pada loss binary cross-entropy MLP.

## Hiperparameter dan Nama Tampilan

Hiperparameter tetap, search space kecil untuk tuning dengan inner validation, dan nama yang ditampilkan di tabel dan gambar. Setiap algoritma memiliki search space berisi tiga atau empat nilai yang mencakup nilai tetapnya, sehingga tuning selalu bisa mempertahankan pengaturan tetap. XGBoost mencari kedalaman pohon dan MLP mencari ukuran hidden layer.

In [ ]:
BASE_PARAMS = {
    "cnb": {"alpha": 1.0},
    "svm": {"C": 1.0, "max_iter": 5000, "random_state": RANDOM_STATE},
    "xgb": {
        "n_estimators": 300,
        "learning_rate": 0.1,
        "max_depth": 6,
        "subsample": 0.8,
        "colsample_bytree": 0.5,
        "min_child_weight": 1,
        "tree_method": "hist",
        "random_state": RANDOM_STATE,
        "n_jobs": 4,
        "verbosity": 0,
    },
    "mlp": {
        "hidden": 256,
        "dropout": 0.3,
        "lr": 1e-3,
        "weight_decay": 1e-5,
        "batch_size": 128,
        "max_epochs": 30,
        "patience": 3,
        "val_fraction": 0.1,
        "random_state": RANDOM_STATE,
    },
    "rf": {
        "n_estimators": 100,
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
    },  # hanya untuk baseline Label Powerset
}
HYPERPARAMETER_GRID = {
    "cnb": [{"alpha": alpha} for alpha in (0.1, 0.3, 1.0)],
    "svm": [{"C": c_value} for c_value in (0.1, 0.3, 1.0, 3.0)],
    "xgb": [{"max_depth": depth} for depth in (4, 6, 8)],
    "mlp": [{"hidden": hidden_size} for hidden_size in (128, 256, 512)],
    "rf": [{}],
}
ALGO_NAMES = {
    "cnb": "Complement NB",
    "svm": "LinearSVC",
    "xgb": "XGBoost",
    "mlp": "MLP",
    "rf": "Random Forest",
    "tf": "Transformer (revised protocol)",
    "ens": "Transformer ensemble",
}
IMBALANCE_NAMES = {
    "none": "No treatment",
    "weight": "Class weighting",
    "mlsmote": "MLSMOTE + repair",
    "mlsmote_raw": "MLSMOTE (original)",
    "mlros": "ML-ROS",
}

## Batch MLP

Menghasilkan baris matriks sparse per batch sebagai tensor dense di device.

In [ ]:
def mlp_batches(X, idx, batch_size):
    for start in range(0, len(idx), batch_size):
        batch_idx = idx[start : start + batch_size]
        batch_features = X[batch_idx]
        batch_features = (
            batch_features.toarray() if sp.issparse(batch_features) else np.asarray(batch_features)
        )
        yield batch_idx, torch.from_numpy(batch_features.astype(np.float32)).to(
            DEVICE, non_blocking=True
        )

## Pembagian Data Validasi MLP

Menyisihkan sebagian baris latih secara terstratifikasi untuk early stopping.

In [ ]:
def mlp_validation_split(labels, val_fraction, rng):
    pos, neg = np.where(labels == 1)[0], np.where(labels == 0)[0]
    n_val_pos = max(1, int(round(len(pos) * val_fraction)))
    n_val_neg = max(1, int(round(len(neg) * val_fraction)))
    val_idx = np.concatenate([rng.permutation(pos)[:n_val_pos], rng.permutation(neg)[:n_val_neg]])
    return np.setdiff1d(np.arange(len(labels)), val_idx), val_idx

## Satu Epoch Training MLP

Satu kali lintasan atas baris latih dengan urutan acak.

In [ ]:
def mlp_train_epoch(model, X, y_tensor, fit_idx, rng, batch_size, loss_fn, optimizer):
    model.train()
    for batch_idx, batch_features in mlp_batches(X, rng.permutation(fit_idx), batch_size):
        optimizer.zero_grad(set_to_none=True)
        loss = loss_fn(model(batch_features).squeeze(1), y_tensor[batch_idx])
        loss.backward()
        optimizer.step()

## Loss Validasi MLP

Rata-rata loss pada baris yang disisihkan.

In [ ]:
def mlp_validation_loss(model, X, y_tensor, val_idx, batch_size, loss_fn):
    model.eval()
    total = 0.0
    with torch.no_grad():
        for batch_idx, batch_features in mlp_batches(X, val_idx, batch_size):
            total += loss_fn(model(batch_features).squeeze(1), y_tensor[batch_idx]).item() * len(
                batch_idx
            )
    return total / len(val_idx)

## Latih MLP

Melatih MLP dengan satu hidden layer untuk satu label biner. Training berhenti jika loss validasi tidak membaik selama sejumlah epoch, dan bobot terbaik dipertahankan.

In [ ]:
def train_mlp(
    X,
    labels,
    weight=False,
    hidden=256,
    dropout=0.3,
    lr=1e-3,
    weight_decay=1e-5,
    batch_size=128,
    max_epochs=30,
    patience=3,
    val_fraction=0.1,
    random_state=RANDOM_STATE,
):
    torch.manual_seed(random_state)
    if DEVICE == "cuda":
        torch.cuda.manual_seed_all(random_state)
    rng = np.random.default_rng(random_state)
    X = sp.csr_matrix(X, dtype=np.float32)
    labels = np.asarray(labels).astype(np.float32)
    fit_idx, val_idx = mlp_validation_split(labels, val_fraction, rng)
    model = nn.Sequential(
        nn.Linear(X.shape[1], hidden), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden, 1)
    ).to(DEVICE)
    pos_weight = None
    if weight:
        n_pos = max(labels[fit_idx].sum(), 1.0)
        pos_weight = torch.tensor([(len(fit_idx) - n_pos) / n_pos], device=DEVICE)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    y_tensor = torch.from_numpy(labels).to(DEVICE)
    best_loss, best_state, bad_epochs = np.inf, None, 0
    for _ in range(max_epochs):
        mlp_train_epoch(model, X, y_tensor, fit_idx, rng, batch_size, loss_fn, optimizer)
        val_loss = mlp_validation_loss(model, X, y_tensor, val_idx, batch_size, loss_fn)
        if val_loss < best_loss - 1e-4:
            best_loss, bad_epochs = val_loss, 0
            best_state = {
                key: tensor.detach().clone() for key, tensor in model.state_dict().items()
            }
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model

## Probabilitas MLP

Probabilitas prediksi kelas positif untuk setiap baris.

In [ ]:
def mlp_probabilities(model, X, batch_size):
    X = sp.csr_matrix(X, dtype=np.float32)
    model.eval()
    out = []
    with torch.no_grad():
        for _, batch_features in mlp_batches(X, np.arange(X.shape[0]), batch_size):
            out.append(torch.sigmoid(model(batch_features).squeeze(1)).float().cpu().numpy())
    return np.concatenate(out)

## Buat Estimator

Membuat model scikit-learn atau XGBoost untuk sebuah algoritma, dengan class weighting jika diminta.

In [ ]:
def make_estimator(algo, weight, labels, params):
    params = dict(params)
    if algo == "cnb":
        return ComplementNB(**params)
    if algo == "svm":
        if weight:
            params["class_weight"] = "balanced"
        return LinearSVC(**params)
    if algo == "xgb":
        if weight:
            pos = max(int(labels.sum()), 1)
            params["scale_pos_weight"] = (len(labels) - pos) / pos
        return XGBClassifier(device=DEVICE, **params)
    if algo == "rf":
        return RandomForestClassifier(**params)
    raise ValueError(algo)

## Latih Model Biner

Melatih satu model biner untuk satu label. Label yang konstan pada data latih menjadi prediksi konstan, dan MLP dikembalikan bersama batch size-nya.

In [ ]:
def train_binary_model(algo, X, labels, weight, override=None):
    if labels.min() == labels.max():
        return ("constant", int(labels[0]))
    params = {**BASE_PARAMS[algo], **(override or {})}
    if algo == "mlp":
        return ("mlp", train_mlp(X, labels, weight=weight, **params), params["batch_size"])
    estimator = make_estimator(algo, weight, labels, params)
    if algo == "xgb":
        X = sp.csr_matrix(X, dtype=np.float32)
    if weight and algo == "cnb":
        estimator.fit(X, labels, sample_weight=compute_sample_weight("balanced", labels))
    else:
        estimator.fit(X, labels)
    return estimator

## Skor Biner

Decision score dari satu model biner, dengan skor positif berarti label diprediksi pada threshold default. LinearSVC memakai decision function-nya, sedangkan model lain memakai probabilitas dikurangi 0,5.

In [ ]:
def binary_scores(model, X):
    if isinstance(model, tuple) and model[0] == "constant":
        return np.full(X.shape[0], 1.0 if model[1] == 1 else -1.0)
    if isinstance(model, tuple) and model[0] == "mlp":
        return mlp_probabilities(model[1], X, model[2]) - 0.5
    if isinstance(model, LinearSVC):
        return model.decision_function(X)
    if isinstance(model, XGBClassifier):
        X = sp.csr_matrix(X, dtype=np.float32)
    return model.predict_proba(X)[:, 1] - 0.5

# Strategi Multi-Label

Binary Relevance melatih satu model biner independen per label. Classifier Chains (Read et al., 2011) memberi setiap model prediksi untuk label-label sebelumnya, dan chain-nya mengikuti urutan LABELS. Label Powerset memperlakukan setiap kombinasi label sebagai satu kelas dan hanya dipakai untuk perbandingan dengan baseline Ibrohim dan Budi (2019).

## Binary Relevance

Satu model biner per label yang dilatih secara independen.

In [ ]:
def fit_binary_relevance(algo, X, Y, weight, override=None):
    return [
        train_binary_model(algo, X, Y[:, label_index], weight, override)
        for label_index in range(Y.shape[1])
    ]

## Tambahkan Label Sebelumnya

Menambahkan label dari link sebelumnya di chain ke matriks fitur.

In [ ]:
def with_previous_labels(X, Y_previous):
    if Y_previous.shape[1] == 0:
        return sp.csr_matrix(X)
    return sp.hstack([sp.csr_matrix(X), sp.csr_matrix(Y_previous.astype(np.float64))], format="csr")

## Classifier Chains

Setiap model dalam chain dilatih dengan fitur ditambah nilai sebenarnya dari label-label sebelumnya.

In [ ]:
def fit_classifier_chain(algo, X, Y, weight, override=None):
    return [
        train_binary_model(
            algo, with_previous_labels(X, Y[:, :label_index]), Y[:, label_index], weight, override
        )
        for label_index in range(Y.shape[1])
    ]

## Skor Label dalam Chain

Skor satu label dalam chain, berdasarkan prediksi untuk label-label sebelumnya.

In [ ]:
def chain_label_scores(models, label_index, X, previous_predictions):
    return binary_scores(models[label_index], with_previous_labels(X, previous_predictions))

## Label Powerset

Setiap kombinasi label yang muncul saat training menjadi satu kelas. Mengembalikan model terlatih dan label set setiap kelas.

In [ ]:
def fit_label_powerset(algo, X, Y, override=None):
    keys = ["".join(map(str, row)) for row in Y]
    classes, class_index = np.unique(keys, return_inverse=True)
    codes = np.array([[int(digit) for digit in key] for key in classes])
    params = {**BASE_PARAMS[algo], **(override or {})}
    estimator_class = {"cnb": ComplementNB, "svm": LinearSVC, "rf": RandomForestClassifier}[algo]
    return estimator_class(**params).fit(X, class_index), codes

## Latih Model Multi-Label

Melatih model dengan strategi yang dipilih dan menyimpan nama strateginya bersama model tersebut.

In [ ]:
def fit_multilabel(strategy, algo, X, Y, weight, override=None):
    if strategy == "br":
        return {"strategy": "br", "models": fit_binary_relevance(algo, X, Y, weight, override)}
    if strategy == "cc":
        return {"strategy": "cc", "models": fit_classifier_chain(algo, X, Y, weight, override)}
    if strategy == "lp":
        return {"strategy": "lp", "models": fit_label_powerset(algo, X, Y, override)}
    raise ValueError(strategy)

## Decision Score

Satu skor per label, dengan skor positif berarti label diprediksi. Threshold dikurangkan dari skor. Pada Classifier Chains, setiap prediksi menjadi input bagi link berikutnya, sehingga threshold diterapkan di sepanjang chain.

In [ ]:
def decision_scores(model, X, thresholds=None):
    models = model["models"]
    if model["strategy"] == "lp":
        estimator, codes = models
        return np.where(codes[estimator.predict(X)] == 1, 1.0, -1.0)
    if model["strategy"] == "br":
        scores = np.column_stack([binary_scores(binary_model, X) for binary_model in models])
        return scores if thresholds is None else scores - np.asarray(thresholds)[None, :]
    label_thresholds = np.zeros(len(models)) if thresholds is None else np.asarray(thresholds)
    scores = np.zeros((X.shape[0], len(models)))
    predictions = np.zeros((X.shape[0], len(models)), dtype=int)
    for label_index in range(len(models)):
        scores[:, label_index] = (
            chain_label_scores(models, label_index, X, predictions[:, :label_index])
            - label_thresholds[label_index]
        )
        predictions[:, label_index] = (scores[:, label_index] > 0).astype(int)
    return scores

# Tuning dengan Inner Validation

Di dalam bagian latih setiap outer fold, 15 persen data disisihkan sebagai inner validation dengan stratifikasi multi-label. Sekumpulan kecil hiperparameter dan satu threshold per label dipilih di sana, lalu model dilatih ulang pada seluruh outer training fold.

## Pembagian Inner Validation

Membagi data latih menjadi bagian untuk melatih model dan bagian inner validation sebesar 15 persen dengan stratifikasi multi-label.

In [ ]:
INNER_VALIDATION_SIZE = 0.15


def inner_validation_split(Y, seed=RANDOM_STATE):
    splitter = MultilabelStratifiedShuffleSplit(
        n_splits=1, test_size=INNER_VALIDATION_SIZE, random_state=seed
    )
    return next(splitter.split(np.zeros(len(Y)), Y))

## Threshold F1 Optimal

Mencari di antara 100 kandidat threshold, yaitu persentil ke-1 hingga ke-99 dari skor ditambah nol, dan mempertahankan threshold dengan F1 tertinggi. Di antara kandidat yang sama baiknya, kandidat yang paling dekat ke nol, yaitu threshold default, yang dipertahankan.

In [ ]:
def f1_optimal_threshold(labels, scores):
    labels, scores = np.asarray(labels), np.asarray(scores)
    if labels.sum() == 0:
        return 0.0
    candidates = np.unique(
        np.concatenate([np.quantile(scores, np.linspace(0.01, 0.99, 99)), [0.0]])
    )
    predicted = scores[None, :] > candidates[:, None]
    tp = (predicted & (labels[None, :] == 1)).sum(1)
    fp = (predicted & (labels[None, :] == 0)).sum(1)
    fn = labels.sum() - tp
    f1 = np.where(2 * tp + fp + fn > 0, 2 * tp / np.maximum(2 * tp + fp + fn, 1), 0)
    best = candidates[np.abs(f1 - f1.max()) <= 1e-12]
    return float(best[np.argmin(np.abs(best))])

## Tuning Threshold per Label

Satu threshold per label pada decision score model terlatih. Untuk Classifier Chains, threshold dipilih sesuai urutan chain, karena setiap prediksi menjadi input bagi link berikutnya.

In [ ]:
def tune_label_thresholds(model, X_val, Y_val):
    n_labels = Y_val.shape[1]
    if model["strategy"] != "cc":
        scores = decision_scores(model, X_val)
        return np.array(
            [
                f1_optimal_threshold(Y_val[:, label_index], scores[:, label_index])
                for label_index in range(n_labels)
            ]
        )
    thresholds = np.zeros(n_labels)
    predictions = np.zeros_like(Y_val)
    for label_index in range(n_labels):
        label_scores = chain_label_scores(
            model["models"], label_index, X_val, predictions[:, :label_index]
        )
        thresholds[label_index] = f1_optimal_threshold(Y_val[:, label_index], label_scores)
        predictions[:, label_index] = (label_scores > thresholds[label_index]).astype(int)
    return thresholds

## Tuning Threshold Probabilitas

Pencarian threshold yang sama pada skala probabilitas, dipakai untuk transformer.

In [ ]:
def tune_probability_thresholds(Y, probabilities):
    return (
        np.array(
            [
                f1_optimal_threshold(Y[:, label_index], probabilities[:, label_index] - 0.5)
                for label_index in range(Y.shape[1])
            ]
        )
        + 0.5
    )

## Pilih dengan Inner Validation

Mencoba setiap kandidat hiperparameter pada bagian inner validation, melakukan tuning threshold untuk masing-masing, dan mempertahankan kandidat dengan macro F1 inner validation tertinggi. Tanpa pencarian hiperparameter, hanya hiperparameter tetap yang dicoba, sehingga hanya threshold yang di-tuning. Mengembalikan hiperparameter terpilih, threshold, dan log pencariannya.

In [ ]:
def select_on_inner_validation(
    algo, strategy, condition, texts_train, Y_train, features="word", tune_hyperparameters=True
):
    i_fit, i_val = inner_validation_split(Y_train)
    vectorizers = fit_features(texts_train.iloc[i_fit], features)
    X_fit = transform_features(vectorizers, texts_train.iloc[i_fit])
    X_val = transform_features(vectorizers, texts_train.iloc[i_val])
    Y_fit, Y_val = Y_train[i_fit], Y_train[i_val]
    if condition in RESAMPLING_CONDITIONS:
        X_fit, Y_fit, _ = resample(X_fit, Y_fit, condition)
    log, best = [], None
    for override in HYPERPARAMETER_GRID[algo] if tune_hyperparameters else [{}]:
        model = fit_multilabel(strategy, algo, X_fit, Y_fit, condition == "weight", override)
        thresholds = (
            np.zeros(Y_train.shape[1])
            if strategy == "lp"
            else tune_label_thresholds(model, X_val, Y_val)
        )
        predictions = (decision_scores(model, X_val, thresholds) > 0).astype(int)
        score = f1_score(Y_val, predictions, average="macro", zero_division=0)
        log.append({"override": override, "inner_macro_f1": round(float(score), 4)})
        if best is None or score > best[0]:
            best = (score, override, thresholds)
    return best[1], best[2], log

# Metrik Evaluasi

Metrik berbasis label adalah macro F1, micro F1, macro F1 atas lima label minoritas, serta precision, recall, dan F1 untuk setiap label. Metrik example-based adalah Hamming loss, indeks Jaccard, dan subset accuracy. Indeks Jaccard per dokumen sama dengan Persamaan 1 pada Ibrohim dan Budi (2019), sehingga dapat dibandingkan dengan baseline mereka sebesar 66,12 persen. Dokumen tanpa label sebenarnya dan tanpa label prediksi dihitung benar. Hierarchy violation rate adalah proporsi label set prediksi yang melanggar aturan anotasi.

## Evaluasi Prediksi

Menghitung semua metrik untuk satu himpunan prediksi.

In [ ]:
def evaluate_predictions(Y_true, Y_pred):
    Y_true = np.asarray(Y_true).astype(int)
    Y_pred = np.asarray(Y_pred).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(
        Y_true, Y_pred, average=None, zero_division=0
    )
    out = {
        "macro_f1": f1_score(Y_true, Y_pred, average="macro", zero_division=0),
        "micro_f1": f1_score(Y_true, Y_pred, average="micro", zero_division=0),
        "minority_macro_f1": float(
            np.mean([f1[minority_index] for minority_index in MINORITY_INDICES])
        ),
        "hamming_loss": hamming_loss(Y_true, Y_pred),
        "jaccard_acc": jaccard_score(Y_true, Y_pred, average="samples", zero_division=1),
        "subset_acc": accuracy_score(Y_true, Y_pred),
        "pred_hierarchy_violation_rate": float(hierarchy_violation_mask(Y_pred).mean()),
    }
    for label_index, label in enumerate(LABELS):
        out[f"f1__{label}"] = f1[label_index]
        out[f"p__{label}"] = precision[label_index]
        out[f"r__{label}"] = recall[label_index]
    return out

## Macro F1 pada Threshold

Macro F1 cepat untuk probabilitas dan threshold, dengan label tanpa positif sebenarnya maupun positif prediksi dihitung nol.

In [ ]:
def macro_f1_at(Y, probabilities, thresholds):
    predicted = probabilities > thresholds
    tp = (predicted & (Y == 1)).sum(0)
    fp = (predicted & (Y == 0)).sum(0)
    fn = (~predicted & (Y == 1)).sum(0)
    return float(
        np.mean(np.where(2 * tp + fp + fn > 0, 2 * tp / np.maximum(2 * tp + fp + fn, 1), 0))
    )

# Uji Statistik

Uji ini mengikuti praktik umum untuk membandingkan model pada satu dataset (Dror et al., 2018; Nadeau dan Bengio, 2003). Setiap perbandingan terdiri atas dua bagian. Paired permutation test memakai prediksi out-of-fold dari seluruh 12.773 tweet, sehingga unitnya adalah tweet. Uji ini menunjukkan apakah dua model berbeda lebih besar daripada yang dapat dijelaskan oleh variasi data uji. Corrected resampled t-test memakai lima nilai macro F1 per fold, sehingga juga mencerminkan variasi yang berasal dari data latih. Uji ini memberikan interval kepercayaan 95 persen untuk selisihnya dan equivalence test terhadap margin yang ditetapkan di konfigurasi. Koreksi Holm-Bonferroni diterapkan di dalam setiap family perbandingan. Ke-12 label tidak dipakai sebagai unit statistik karena saling bergantung melalui hierarki.

Sebuah perbandingan disebut better jika kedua uji signifikan setelah koreksi, equivalent jika equivalence test signifikan, dan inconclusive jika tidak memenuhi keduanya.

## Count Matrix

True positive, false positive, dan false negative untuk setiap tweet dan label. Jumlahnya memberikan hitungan di balik macro F1, sehingga permutation test berjalan cepat.

In [ ]:
def count_matrices(Y, pred):
    Y, pred = np.asarray(Y).astype(bool), np.asarray(pred).astype(bool)
    return np.stack([pred & Y, pred & ~Y, ~pred & Y], axis=1).astype(np.float32)

## Macro F1 dari Count

Macro F1 dari count yang dijumlahkan, untuk satu atau banyak himpunan count sekaligus. Label tanpa positif sebenarnya maupun positif prediksi dihitung nol.

In [ ]:
def macro_f1_from_counts(totals):
    tp, fp, fn = totals[..., 0, :], totals[..., 1, :], totals[..., 2, :]
    denominator = 2 * tp + fp + fn
    return np.where(denominator > 0, 2 * tp / np.maximum(denominator, 1), 0).mean(axis=-1)

## Paired Permutation Test

Di bawah hipotesis nol, kedua model dapat saling dipertukarkan, sehingga prediksi kedua model ditukar secara acak untuk setiap tweet. p-value adalah proporsi pertukaran acak yang selisih macro F1-nya minimal sebesar selisih yang teramati.

In [ ]:
def permutation_test_macro_f1(
    Y, pred_a, pred_b, n_permutations=N_PERMUTATIONS, seed=RANDOM_STATE, chunk=500
):
    counts_a, counts_b = count_matrices(Y, pred_a), count_matrices(Y, pred_b)
    total_a, total_b = counts_a.sum(axis=0), counts_b.sum(axis=0)
    observed = macro_f1_from_counts(total_b) - macro_f1_from_counts(total_a)
    difference = (counts_b - counts_a).reshape(len(counts_a), -1)
    rng = np.random.default_rng(seed)
    n_extreme = 0
    for start in range(0, n_permutations, chunk):
        size = min(chunk, n_permutations - start)
        swap = (rng.random((size, len(counts_a))) < 0.5).astype(np.float32)
        shift = (swap @ difference).reshape(size, *total_a.shape)
        statistic = macro_f1_from_counts(total_b - shift) - macro_f1_from_counts(total_a + shift)
        n_extreme += int((np.abs(statistic) >= abs(observed) - 1e-12).sum())
    return float(observed), (n_extreme + 1) / (n_permutations + 1)

## Interval Corrected Resampled t

Corrected resampled t-test dari Nadeau dan Bengio (2003) pada selisih per fold. Fungsi ini mengembalikan interval kepercayaan 95 persen dari rata-rata selisih, p-value dari selisih tersebut, dan p-value dari equivalence test berupa two one-sided test terhadap margin.

In [ ]:
def corrected_t_interval(differences, n_train, n_test, margin=EQUIVALENCE_MARGIN):
    fold_differences = np.asarray(differences, float)
    n_folds, mean = len(fold_differences), fold_differences.mean()
    standard_error = np.sqrt((1 / n_folds + n_test / n_train) * fold_differences.var(ddof=1))
    if standard_error == 0:
        return {
            "low": mean,
            "high": mean,
            "p": 1.0 if mean == 0 else 0.0,
            "p_equivalence": 0.0 if abs(mean) < margin else 1.0,
        }
    half_width = stats.t.ppf(0.975, n_folds - 1) * standard_error
    p_equivalence = max(
        stats.t.sf((mean + margin) / standard_error, n_folds - 1),
        stats.t.cdf((mean - margin) / standard_error, n_folds - 1),
    )
    return {
        "low": mean - half_width,
        "high": mean + half_width,
        "p": float(2 * stats.t.sf(abs(mean) / standard_error, n_folds - 1)),
        "p_equivalence": float(p_equivalence),
    }

## Koreksi Holm-Bonferroni

Menyesuaikan daftar p-value untuk multiple comparison.

In [ ]:
def holm_bonferroni(pvals):
    p_values = np.asarray(pvals, float)
    order = np.argsort(p_values)
    adjusted = np.empty(len(p_values))
    running = 0.0
    for rank, index in enumerate(order):
        running = max(running, (len(p_values) - rank) * p_values[index])
        adjusted[index] = min(running, 1.0)
    return adjusted

## Tambahkan Kolom Holm

Menambahkan salinan terkoreksi dari setiap kolom p-value, dikoreksi di dalam setiap kelompok pada sebuah kolom atau atas semua baris.

In [ ]:
def add_holm_columns(table, p_columns, by=None):
    groups = (
        [np.ones(len(table), dtype=bool)]
        if by is None
        else [(table[by] == group).to_numpy() for group in table[by].unique()]
    )
    for mask in groups:
        for col in p_columns:
            table.loc[mask, f"{col}_holm"] = holm_bonferroni(table.loc[mask, col].to_numpy())
    return table

## Bandingkan Dua Model

Membandingkan model B dengan model A. Selisih dan intervalnya berasal dari macro F1 per fold, sedangkan permutation test berasal dari prediksi out-of-fold. Jumlah label tempat B lebih baik adalah deskripsi, bukan uji.

In [ ]:
def compare_models(Y, pred_a, pred_b, folds_a, folds_b, n_train, n_test):
    folds_a, folds_b = np.asarray(folds_a, float), np.asarray(folds_b, float)
    interval = corrected_t_interval(folds_b - folds_a, n_train, n_test)
    _, p_permutation = permutation_test_macro_f1(Y, pred_a, pred_b)
    f1_a = f1_score(Y, pred_a, average=None, zero_division=0)
    f1_b = f1_score(Y, pred_b, average=None, zero_division=0)
    return {
        "macro_f1_a": folds_a.mean(),
        "macro_f1_b": folds_b.mean(),
        "delta_macro_f1": (folds_b - folds_a).mean(),
        "ci95_low": interval["low"],
        "ci95_high": interval["high"],
        "n_labels_improved": int((f1_b > f1_a).sum()),
        "p_permutation": p_permutation,
        "p_corrected_t": interval["p"],
        "p_equivalence": interval["p_equivalence"],
    }

## Tambahkan Verdict

Mengoreksi ketiga p-value dengan Holm-Bonferroni dan memberi setiap perbandingan sebuah verdict: B better, A better, equivalent, atau inconclusive.

In [ ]:
def add_verdicts(table, by=None):
    table = add_holm_columns(table, ["p_permutation", "p_corrected_t", "p_equivalence"], by=by)
    significant = (table.p_permutation_holm < 0.05) & (table.p_corrected_t_holm < 0.05)
    conditions = [
        significant & (table.delta_macro_f1 > 0),
        significant & (table.delta_macro_f1 < 0),
        table.p_equivalence_holm < 0.05,
    ]
    table["verdict"] = np.select(
        conditions, ["B better", "A better", "equivalent"], default="inconclusive"
    )
    return table

## Warna Verdict

Warna setiap verdict pada grafik perbandingan.

In [ ]:
VERDICT_COLORS = {
    "B better": "#2e86c1",
    "A better": "#c0392b",
    "equivalent": "#27ae60",
    "inconclusive": "#7f8c8d",
}

## Plot Perbandingan

Menggambar selisih macro F1 beserta interval 95 persennya. Pita hijau adalah equivalence margin, dan warnanya menunjukkan verdict.

In [ ]:
def plot_comparisons(comparisons, label_col, title, ax):
    positions = np.arange(len(comparisons))[::-1]
    ax.axvspan(-EQUIVALENCE_MARGIN, EQUIVALENCE_MARGIN, color="#d5f5e3", zorder=0)
    ax.axvline(0, color="black", lw=0.8)
    for position, (_, row) in zip(positions, comparisons.iterrows()):
        error = [[row.delta_macro_f1 - row.ci95_low], [row.ci95_high - row.delta_macro_f1]]
        ax.errorbar(
            row.delta_macro_f1,
            position,
            xerr=error,
            fmt="o",
            ms=4,
            capsize=2,
            color=VERDICT_COLORS[row.verdict],
        )
    ax.set_yticks(positions)
    ax.set_yticklabels(comparisons[label_col], fontsize=7)
    ax.set_xlabel("Difference in macro F1, B minus A, with 95% interval", fontsize=8)
    ax.set_title(title, fontsize=9)

# Runner Eksperimen

Satu run adalah satu kombinasi algoritma, strategi, penanganan ketidakseimbangan, tuning, fitur, dan fold. Baris ditambahkan ke results/experiments.csv untuk grid utama atau ke results/experiments_ext.csv untuk eksperimen lanjutan. Decision score pada test fold disimpan di results/preds atau results/preds_ext. Run yang sudah ada di file hasil dilewati. Model CPU berjalan paralel dan model GPU berjalan satu per satu.

## Susunan File Hasil

Urutan kolom kedua log run, dipakai ketika file log dibuat dari awal.

In [ ]:
GPU_ALGOS = {"xgb", "mlp"}
METRIC_COLUMNS = [
    "macro_f1",
    "micro_f1",
    "minority_macro_f1",
    "hamming_loss",
    "jaccard_acc",
    "subset_acc",
    "pred_hierarchy_violation_rate",
] + [f"{metric}__{label}" for label in LABELS for metric in ("f1", "p", "r")]
RESULT_COLUMNS = {
    RESULTS_CSV: [
        "config",
        "algo",
        "strategy",
        "condition",
        "tuned",
        "fold",
        "n_train",
        "n_test",
        "device",
        "train_time_s",
        "tune_time_s",
        "total_time_s",
        "n_synthetic",
        "n_repaired",
        "override",
        "thresholds",
        "inner_log",
    ]
    + METRIC_COLUMNS,
    EXTENDED_CSV: [
        "config",
        "algo",
        "strategy",
        "condition",
        "features",
        "tuned",
        "fold",
        "n_train",
        "n_test",
        "device",
        "train_time_s",
        "tune_time_s",
        "total_time_s",
        "n_synthetic",
        "override",
        "thresholds",
        "inner_log",
        "inner_macro_f1",
        "best_epoch",
    ]
    + METRIC_COLUMNS,
}

## ID Konfigurasi

Menyusun identitas sebuah run, misalnya svm__br__none__tuned. Fitur hanya ditambahkan jika bukan word, dan akhirannya menunjukkan mode tuning: tanpa akhiran untuk fixed, thr untuk threshold saja, dan tuned untuk hiperparameter dan threshold.

In [ ]:
TUNING_SUFFIX = {"fixed": "", "thr": "__thr", "tuned": "__tuned"}


def config_id(algo, strategy, condition, tuning="fixed", features="word"):
    feature_part = "" if features == "word" else f"__{features}"
    return f"{algo}__{strategy}__{condition}{feature_part}{TUNING_SUFFIX[tuning]}"

## Muat Input

Membaca data hasil deduplikasi dan fold yang tersimpan.

In [ ]:
def load_inputs(root):
    data = pd.read_csv(root / "data" / "processed" / "data_clean.csv", keep_default_na=False)
    folds = json.loads((root / "results" / "folds.json").read_text())
    return data, folds

## Baris Fold

Mengubah id latih dan id uji dari satu fold menjadi posisi baris.

In [ ]:
def fold_rows(data, fold):
    row_of_id = pd.Series(np.arange(len(data)), index=data["id"])
    return row_of_id[fold["train_ids"]].to_numpy(), row_of_id[fold["test_ids"]].to_numpy()

## Run yang Selesai

Konfigurasi dan fold dari setiap run yang sudah tercatat di log run.

In [ ]:
def completed_runs(path):
    if not path.exists():
        return set()
    previous = pd.read_csv(path)
    return set(zip(previous.config, previous.fold))

## Tambahkan Baris

Menambahkan baris hasil ke log run dengan mempertahankan urutan kolom file yang sudah ada.

In [ ]:
def append_rows(path, rows):
    table = pd.DataFrame(rows)
    if path.exists():
        columns = pd.read_csv(path, nrows=0).columns.tolist()
    else:
        columns = RESULT_COLUMNS.get(path.name, table.columns.tolist())
    table.reindex(columns=columns).to_csv(path, mode="a", header=not path.exists(), index=False)

## Latih dan Beri Skor

Melatih fitur pada training fold, melakukan resampling pada training fold jika penanganan ketidakseimbangan memintanya, melatih model, dan memberi skor pada test fold. Tidak ada yang dilatih pada test fold.

In [ ]:
def train_and_score(
    algo, strategy, condition, override, thresholds, texts_train, Y_train, texts_test, features
):
    vectorizers = fit_features(texts_train, features)
    X_train = transform_features(vectorizers, texts_train)
    X_test = transform_features(vectorizers, texts_test)
    resample_summary = {}
    if condition in RESAMPLING_CONDITIONS:
        X_train, Y_train, resample_summary = resample(X_train, Y_train, condition)
    start_time = time.time()
    model = fit_multilabel(strategy, algo, X_train, Y_train, condition == "weight", override)
    train_time = time.time() - start_time
    return (
        decision_scores(model, X_test, thresholds),
        X_train.shape[0],
        train_time,
        resample_summary,
    )

## Jalankan Satu Eksperimen

Menjalankan satu konfigurasi pada satu fold dan mengembalikan baris hasil. Mode tuning fixed tidak memakai inner validation, thr hanya melakukan tuning threshold, dan tuned juga mencari hiperparameter. Skor uji disimpan untuk analisis selanjutnya.

In [ ]:
def run_experiment(
    root, algo, strategy, condition, tuning, features, fold_id, data, folds, preds_subdir="preds"
):
    train_rows, test_rows = fold_rows(data, folds[fold_id])
    Y, texts = label_matrix(data), data["text_clean"]
    start_time = time.time()
    override, thresholds, tuning_log = None, None, []
    if tuning != "fixed":
        override, thresholds, tuning_log = select_on_inner_validation(
            algo,
            strategy,
            condition,
            texts.iloc[train_rows].reset_index(drop=True),
            Y[train_rows],
            features,
            tune_hyperparameters=(tuning == "tuned"),
        )
    tune_time = time.time() - start_time
    scores, n_train, train_time, resample_summary = train_and_score(
        algo,
        strategy,
        condition,
        override,
        thresholds,
        texts.iloc[train_rows],
        Y[train_rows],
        texts.iloc[test_rows],
        features,
    )
    row = {
        "config": config_id(algo, strategy, condition, tuning, features),
        "algo": algo,
        "strategy": strategy,
        "condition": condition,
        "features": features,
        "tuned": tuning != "fixed",
        "fold": fold_id,
        "n_train": n_train,
        "n_test": len(test_rows),
        "device": DEVICE_LABEL if algo in GPU_ALGOS else "cpu",
        "train_time_s": round(train_time, 2),
        "tune_time_s": round(tune_time, 2),
        "total_time_s": round(time.time() - start_time, 2),
        "n_synthetic": resample_summary.get("n_synthetic", 0),
        "n_repaired": resample_summary.get("n_repaired", 0),
        "override": json.dumps(override or {}),
        "thresholds": (
            json.dumps([round(float(threshold), 4) for threshold in thresholds])
            if thresholds is not None
            else ""
        ),
        "inner_log": json.dumps(tuning_log),
        "inner_macro_f1": max(
            [candidate["inner_macro_f1"] for candidate in tuning_log], default=np.nan
        ),
    }
    row.update(evaluate_predictions(Y[test_rows], (scores > 0).astype(int)))
    np.savez_compressed(
        root / "results" / preds_subdir / f"{row['config']}__f{fold_id}.npz",
        test_ids=np.asarray(folds[fold_id]["test_ids"]),
        scores=scores.astype(np.float32),
    )
    return row

## Bangun Grid

Mendaftar setiap kombinasi algoritma, strategi, penanganan ketidakseimbangan, mode tuning, fitur, dan fold.

In [ ]:
def build_grid(algos, strategies, conditions, tuning=("fixed",), features=("word",), n_folds=5):
    return [
        (algo, strategy, condition, tuning_mode, feature_set, fold)
        for algo in algos
        for strategy in strategies
        for condition in conditions
        for tuning_mode in tuning
        for feature_set in features
        for fold in range(n_folds)
    ]

## Cetak Ringkasan Run

Mencetak satu baris untuk setiap run yang selesai.

In [ ]:
def print_run(row):
    print(
        f"{row['config']:<36} f{row['fold']} macro-F1={row['macro_f1']:.3f} "
        f"minority={row['minority_macro_f1']:.3f} jaccard={row['jaccard_acc']:.3f} "
        f"{row['total_time_s']:.0f}s [{row['device']}]",
        flush=True,
    )

## Jalankan Grid

Menjalankan setiap entri grid yang belum ada di file hasil. Model CPU berjalan dalam batch paralel dan model GPU berjalan satu per satu.

In [ ]:
def run_grid(root, grid, n_jobs_cpu=2, csv_name=RESULTS_CSV, preds_subdir="preds", verbose=True):
    out = Path(root) / "results" / csv_name
    (Path(root) / "results" / preds_subdir).mkdir(parents=True, exist_ok=True)
    done = completed_runs(out)
    todo = [entry for entry in grid if (config_id(*entry[:5]), entry[5]) not in done]
    print(
        f"{csv_name}: {len(grid)} runs in grid, {len(grid) - len(todo)} done, {len(todo)} to run "
        f"| device: {DEVICE}",
        flush=True,
    )
    if not todo:
        return pd.read_csv(out)
    data, folds = load_inputs(Path(root))

    def job(entry):
        return run_experiment(Path(root), *entry, data=data, folds=folds, preds_subdir=preds_subdir)

    def record(rows):
        append_rows(out, rows)
        if verbose:
            for row in rows:
                print_run(row)

    cpu_runs = [entry for entry in todo if entry[0] not in GPU_ALGOS]
    batch = max(n_jobs_cpu, 1) * 2
    for start in range(0, len(cpu_runs), batch):
        record(
            Parallel(n_jobs=n_jobs_cpu)(
                delayed(job)(entry) for entry in cpu_runs[start : start + batch]
            )
        )
    for entry in [entry for entry in todo if entry[0] in GPU_ALGOS]:
        record([job(entry)])
    return pd.read_csv(out)

## Muat Run Lengkap

Membaca log run, membuang konfigurasi yang tidak memiliki kelima fold, dan menambahkan kolom tuning berisi fixed, thr, atau tuned. Mode threshold saja dikenali dari akhirannya pada ID konfigurasi.

In [ ]:
def tuning_label(runs):
    tuned = np.where(runs.tuned.astype(bool), "tuned", "fixed")
    return np.where(runs.config.str.endswith("__thr"), "thr", tuned)


def load_complete_runs(path):
    runs = pd.read_csv(path)
    n_folds = runs.groupby("config").fold.nunique()
    incomplete = n_folds[n_folds < 5]
    if len(incomplete):
        print(
            f"WARNING: configurations without five folds in {path.name} (ignored):",
            incomplete.to_dict(),
        )
        runs = runs[~runs.config.isin(incomplete.index)].copy()
    runs["tuning"] = tuning_label(runs)
    return runs

## Muat Skor Fold

Mengembalikan id uji dan decision score dari satu fold, dengan skor positif berarti prediksi positif. Run transformer menyimpan probabilitas, sehingga threshold-nya dikurangkan di sini.

In [ ]:
def load_fold_scores(config, fold):
    path = RES / "preds" / f"{config}__f{fold}.npz"
    if path.exists():
        stored = np.load(path)
        return stored["test_ids"], stored["scores"]
    base = config[: -len("__tuned")] if config.endswith("__tuned") else config
    if base.startswith(("tf__", "ens__")):
        stored = np.load(RES / "preds_ext" / f"{base}__f{fold}.npz")
        thresholds = stored["thresholds"] if config.endswith("__tuned") else 0.5
        return (
            stored["test_ids"],
            stored["test_prob"] - np.atleast_1d(np.asarray(thresholds, dtype=float))[None, :],
        )
    stored = np.load(RES / "preds_ext" / f"{config}__f{fold}.npz")
    return stored["test_ids"], stored["scores"]

## Prediksi Out-of-Fold

Menggabungkan prediksi uji dari kelima fold, sehingga setiap tweet memiliki tepat satu prediksi dari model yang tidak melihatnya saat training.

In [ ]:
def oof_predictions(config, data):
    row_of_id = pd.Series(np.arange(len(data)), index=data["id"])
    pred = np.zeros((len(data), len(LABELS)), dtype=int)
    for fold in range(5):
        test_ids, scores = load_fold_scores(config, fold)
        pred[row_of_id[test_ids].to_numpy()] = (scores > 0).astype(int)
    return pred

## Array Model

Prediksi out-of-fold dan macro F1 per fold dari satu konfigurasi, yaitu dua input untuk perbandingan model.

In [ ]:
def model_arrays(config, runs, data):
    return oof_predictions(config, data), per_label_and_folds(runs, config)[1]

## Bandingkan Konfigurasi

Membandingkan konfigurasi B dengan konfigurasi A untuk setiap pasangan yang kedua konfigurasinya memiliki kelima fold. Setiap pasangan adalah dictionary dengan key config_a dan config_b, dan entri lainnya dipertahankan sebagai kolom hasil.

In [ ]:
def compare_configs(pairs, runs, data, n_train, n_test):
    Y_all = label_matrix(data)
    folds_per_config = runs.groupby("config").fold.nunique()
    complete = set(folds_per_config[folds_per_config == 5].index)
    rows = []
    for pair in pairs:
        if pair["config_a"] in complete and pair["config_b"] in complete:
            pred_a, folds_a = model_arrays(pair["config_a"], runs, data)
            pred_b, folds_b = model_arrays(pair["config_b"], runs, data)
            rows.append(
                {**pair, **compare_models(Y_all, pred_a, pred_b, folds_a, folds_b, n_train, n_test)}
            )
    return pd.DataFrame(rows)

## Tampilkan Tabel

Membulatkan tabel, mengganti nama kolom condition menjadi imbalance_treatment, lalu menampilkannya secara lengkap atau hanya baris-baris awalnya.

In [ ]:
COLUMN_RENAMES = {"condition": "imbalance_treatment"}


def show_table(table, decimals=4, head=None):
    out = table.round(decimals) if decimals is not None else table.copy()
    out = out.rename(columns=COLUMN_RENAMES)
    out.index = out.index.set_names([COLUMN_RENAMES.get(name, name) for name in out.index.names])
    display(out.head(head) if head else out)

## Muat Hasil Lanjutan

Membaca log run eksperimen lanjutan, atau tabel kosong jika file tersebut belum ada.

In [ ]:
def load_extended():
    path = RES / EXTENDED_CSV
    return pd.read_csv(path) if path.exists() else pd.DataFrame(columns=["config", "algo", "fold"])

## Muat Semua Run

Kedua log run digabung, tanpa baris dari protokol IndoBERTweet sebelumnya.

In [ ]:
def load_all_runs():
    runs = pd.concat([pd.read_csv(RES / RESULTS_CSV), load_extended()], ignore_index=True)
    return runs[runs.algo != "bert"].reset_index(drop=True)

## Cari Stale Run

Run tuned dianggap stale jika log pencariannya memiliki jumlah kandidat yang berbeda dari search space saat ini, dan run threshold saja dianggap stale jika mencoba lebih dari satu kandidat. Fungsi ini menemukan run XGBoost dan MLP tuned yang dibuat sebelum keduanya memiliki search space.

In [ ]:
def find_stale_runs(path):
    if not path.exists():
        return pd.DataFrame(columns=["config", "fold"])
    runs = pd.read_csv(path)
    tuning = tuning_label(runs)
    n_candidates = runs.inner_log.map(
        lambda log_text: (
            len(json.loads(log_text))
            if isinstance(log_text, str) and log_text.startswith("[")
            else 0
        )
    )
    expected = runs.algo.map(lambda algo: len(HYPERPARAMETER_GRID.get(algo, [])))
    classical = runs.algo.isin(list(HYPERPARAMETER_GRID))
    stale = classical & (
        ((tuning == "tuned") & (n_candidates != expected))
        | ((tuning == "thr") & (n_candidates != 1))
    )
    return runs[stale]

## Hapus Run

Menghapus run tertentu dari log run beserta skor uji tersimpannya, sehingga grid menjalankannya lagi.

In [ ]:
def remove_runs(path, runs_to_remove, preds_subdir):
    drop = set(zip(runs_to_remove.config, runs_to_remove.fold))
    runs = pd.read_csv(path)
    keep = [(config, fold) not in drop for config, fold in zip(runs.config, runs.fold)]
    runs[keep].to_csv(path, index=False)
    for config, fold in drop:
        (RES / preds_subdir / f"{config}__f{fold}.npz").unlink(missing_ok=True)

# Fine-Tuning Transformer

Transformer di-fine-tune dengan satu protokol yang menyisihkan 15 persen dari setiap outer training fold sebagai inner validation. Protokol sebelumnya melatih IndoBERTweet pada teks bersih selama paling banyak empat epoch dan berhenti di batas itu pada setiap fold, sehingga tidak dipakai.

Protokol ini memakai tweet asli dengan normalisasi IndoBERTweet. Epoch dipilih berdasarkan macro F1 inner validation setelah threshold tuning, dengan paling banyak 6 atau 10 epoch dan patience 2. Positive class weight-nya adalah akar kuadrat dari rasio negatif terhadap positif dengan batas atas 10, dan protokol ini menyimpan probabilitas inner validation untuk ensemble dan pemilihan threshold.

## Tetapkan Seed Global

Menetapkan seed untuk Python, NumPy, dan PyTorch.

In [ ]:
def set_global_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

## Encode Teks

Melakukan tokenisasi semua teks sekali, dengan padding ke panjang yang sama.

In [ ]:
def encode_texts(tokenizer, texts, max_len):
    enc = tokenizer(
        list(texts), truncation=True, max_length=max_len, padding="max_length", return_tensors="pt"
    )
    return {
        key: tensor
        for key, tensor in enc.items()
        if key in ("input_ids", "attention_mask", "token_type_ids")
    }

## Prediksi Probabilitas

Output sigmoid untuk baris tertentu dari teks yang sudah di-encode, dihitung per batch dengan mixed precision di GPU.

In [ ]:
def predict_probabilities(model, enc, idx, batch_size, device):
    model.eval()
    out = []
    with torch.inference_mode():
        for start in range(0, len(idx), batch_size):
            batch_idx = torch.as_tensor(idx[start : start + batch_size])
            batch = {key: tensor[batch_idx].to(device) for key, tensor in enc.items()}
            with torch.autocast(
                device_type="cuda", dtype=torch.float16, enabled=(device == "cuda")
            ):
                logits = model(**batch).logits
            out.append(torch.sigmoid(logits.float()).cpu().numpy())
    return np.concatenate(out)

## Optimizer dan Scheduler

Membuat optimizer AdamW, linear schedule dengan warmup, dan gradient scaler untuk mixed precision.

In [ ]:
def make_training_tools(model, n_rows, batch_size, epochs, lr, weight_decay, warmup, device):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    steps = math.ceil(n_rows / batch_size) * epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, int(warmup * steps), steps)
    scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))
    return optimizer, scheduler, scaler

## Latih Satu Epoch

Satu kali lintasan atas baris yang diberikan dengan urutan acak, memakai mixed precision dan gradient clipping. Mengembalikan jumlah loss.

In [ ]:
def train_one_epoch(model, enc, Y_tensor, rows, rng, batch_size, loss_fn, tools, device):
    optimizer, scheduler, scaler = tools
    model.train()
    order = rng.permutation(rows)
    total = 0.0
    for start in range(0, len(order), batch_size):
        batch_idx = torch.as_tensor(order[start : start + batch_size])
        batch = {key: tensor[batch_idx].to(device) for key, tensor in enc.items()}
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device == "cuda")):
            logits = model(**batch).logits
        loss = loss_fn(logits.float(), Y_tensor[batch_idx].to(device))
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total += float(loss.detach())
    return total

## Salin State Model

Salinan bobot model di CPU, dipakai untuk menyimpan epoch terbaik.

In [ ]:
def copy_model_state(model):
    return {key: tensor.detach().cpu().clone() for key, tensor in model.state_dict().items()}

## Positive Class Weight

Bobot untuk kelas positif pada loss binary cross-entropy: akar kuadrat dari rasio contoh negatif terhadap contoh positif, dengan batas atas 10.

In [ ]:
def positive_class_weights(Y, device):
    pos = Y.sum(0).clip(min=1)
    ratio = np.minimum(np.sqrt((len(Y) - pos) / pos), 10.0)
    return torch.tensor(ratio, dtype=torch.float32, device=device)

## Default Protokol

Pengaturan default protokol. Setiap konfigurasi menimpa sebagian di antaranya. Entri text, loss, dan pos_weight mencatat pilihan tetap protokol di log run, yaitu tweet asli, binary cross-entropy, dan class weight akar kuadrat.

In [ ]:
TRANSFORMER_DEFAULTS = {
    "model": "indolem/indobertweet-base-uncased",
    "text": "bert",
    "max_len": 128,
    "lr": 2e-5,
    "epochs": 6,
    "patience": 2,
    "batch": 16,
    "pos_weight": "sqrt10",
    "loss": "bce",
    "seed": RANDOM_STATE,
    "warmup": 0.1,
    "weight_decay": 0.01,
}

## Fine-Tune dengan Pemilihan Epoch

Melakukan fine-tuning dan mempertahankan epoch dengan macro F1 inner validation terbaik setelah threshold tuning. Mengembalikan catatan epoch terbaik, riwayat training, dan waktu training, lalu memuat bobot epoch terbaik ke dalam model.

In [ ]:
def fine_tune_with_epoch_selection(cfg, model, enc, Y, fit_rows, val_rows, device, log, tag):
    Y_tensor = torch.from_numpy(Y)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=positive_class_weights(Y[fit_rows], device))
    tools = make_training_tools(
        model,
        len(fit_rows),
        cfg["batch"],
        cfg["epochs"],
        cfg["lr"],
        cfg["weight_decay"],
        cfg["warmup"],
        device,
    )
    start_time = time.time()
    best, history, bad_epochs = None, [], 0
    rng = np.random.default_rng(cfg["seed"])
    y_val = Y[val_rows].astype(int)
    for epoch in range(cfg["epochs"]):
        total_loss = train_one_epoch(
            model, enc, Y_tensor, fit_rows, rng, cfg["batch"], loss_fn, tools, device
        )
        p_val = predict_probabilities(model, enc, val_rows, cfg["batch"] * 4, device)
        thresholds = tune_probability_thresholds(y_val, p_val)
        f1_tuned, f1_default = macro_f1_at(y_val, p_val, thresholds), macro_f1_at(y_val, p_val, 0.5)
        history.append(
            {
                "epoch": epoch + 1,
                "loss": round(total_loss / math.ceil(len(fit_rows) / cfg["batch"]), 4),
                "inner_f1_tuned": round(f1_tuned, 4),
                "inner_f1_05": round(f1_default, 4),
            }
        )
        log(
            f"  {tag} epoch {epoch + 1}: inner macro-F1 tuned={f1_tuned:.4f} "
            f"(0.5: {f1_default:.4f}) {(time.time() - start_time) / 60:.1f} min"
        )
        if best is None or f1_tuned > best["f"] + 1e-4:
            best, bad_epochs = {
                "f": f1_tuned,
                "epoch": epoch + 1,
                "thr": thresholds,
                "pv": p_val,
                "state": copy_model_state(model),
            }, 0
        else:
            bad_epochs += 1
            if bad_epochs >= cfg["patience"]:
                break
    model.load_state_dict(best["state"])
    return best, history, time.time() - start_time

## Baris Hasil Protokol Revisi

Membuat satu baris dengan threshold 0,5 dan satu baris dengan threshold hasil tuning dari epoch terbaik.

In [ ]:
def revised_result_rows(name, cfg, fold_id, best, history, p_test, Y_test, info):
    rows = []
    for tuned, thresholds in [(False, np.full(len(LABELS), 0.5)), (True, best["thr"])]:
        row = {
            "config": f"tf__{name}" + ("__tuned" if tuned else ""),
            "algo": "tf",
            "strategy": "mo",
            "condition": name,
            "features": cfg["text"],
            "tuned": tuned,
            "fold": fold_id,
            "n_train": info["n_train"],
            "n_test": len(Y_test),
            "device": info["device"],
            "train_time_s": round(info["train_time"], 1),
            "tune_time_s": 0.0,
            "total_time_s": round(time.time() - info["t0"], 1),
            "n_synthetic": 0,
            "override": json.dumps({key: cfg[key] for key in sorted(cfg)}),
            "thresholds": (
                json.dumps([round(float(threshold), 4) for threshold in thresholds])
                if tuned
                else ""
            ),
            "inner_log": json.dumps(history),
            "inner_macro_f1": round(best["f"], 4),
            "best_epoch": best["epoch"],
        }
        row.update(evaluate_predictions(Y_test, (p_test > thresholds[None, :]).astype(int)))
        rows.append(row)
    return rows

## Jalankan Satu Fold Transformer

Menjalankan satu konfigurasi protokol pada satu outer fold. Menambahkan dua baris hasil dan menyimpan probabilitas inner validation dan probabilitas uji.

In [ ]:
def run_transformer_fold(
    root, name, cfg, data, folds, fold_id, device, done, csv_name=EXTENDED_CSV, log=print
):
    settings = {**TRANSFORMER_DEFAULTS, **cfg}
    if (f"tf__{name}__tuned", fold_id) in done:
        log(f"skip tf__{name} f{fold_id} (already done)")
        return
    set_global_seed(settings["seed"])
    train_rows, test_rows = fold_rows(data, folds[fold_id])
    Y = data[LABELS].to_numpy(dtype=np.float32)
    i_fit, i_val = inner_validation_split(Y[train_rows].astype(int))
    fit_rows, val_rows = train_rows[i_fit], train_rows[i_val]
    tokenizer = AutoTokenizer.from_pretrained(settings["model"])
    model = AutoModelForSequenceClassification.from_pretrained(
        settings["model"], num_labels=len(LABELS), problem_type="multi_label_classification"
    ).to(device)
    texts = data["Tweet"].map(normalize_for_transformer)
    enc = encode_texts(tokenizer, texts, settings["max_len"])
    best, history, train_time = fine_tune_with_epoch_selection(
        settings, model, enc, Y, fit_rows, val_rows, device, log, f"tf__{name} f{fold_id}"
    )
    start_time = time.time() - train_time
    p_test = predict_probabilities(model, enc, test_rows, settings["batch"] * 4, device)
    ids = data["id"].to_numpy()
    np.savez_compressed(
        root / "results" / "preds_ext" / f"tf__{name}__f{fold_id}.npz",
        test_ids=ids[test_rows],
        test_prob=p_test.astype(np.float32),
        inner_ids=ids[val_rows],
        inner_prob=best["pv"].astype(np.float32),
        thresholds=best["thr"].astype(np.float32),
    )
    info = {"n_train": len(fit_rows), "device": device, "train_time": train_time, "t0": start_time}
    rows = revised_result_rows(
        name, settings, fold_id, best, history, p_test, Y[test_rows].astype(int), info
    )
    append_rows(root / "results" / csv_name, rows)
    log(
        f"tf__{name} f{fold_id}: best epoch {best['epoch']}, inner {best['f']:.4f}, "
        f"test macro-F1 tuned={rows[1]['macro_f1']:.4f} (0.5: {rows[0]['macro_f1']:.4f}), "
        f"{train_time / 60:.1f} min"
    )
    del model
    if device == "cuda":
        torch.cuda.empty_cache()

## Jalankan Konfigurasi Transformer

Menjalankan setiap konfigurasi protokol pada setiap fold dan melewati fold yang sudah selesai.

In [ ]:
def run_transformer_configs(root, configs, folds_to_run=range(5), csv_name=EXTENDED_CSV, log=print):
    data, folds = load_inputs(root)
    out = root / "results" / csv_name
    for name, cfg in configs.items():
        for fold in folds_to_run:
            run_transformer_fold(
                root, name, cfg, data, folds, fold, DEVICE, completed_runs(out), csv_name, log
            )

## Ensemble Satu Fold

Merata-ratakan probabilitas tersimpan dari anggota-anggotanya pada satu fold dan melakukan tuning threshold pada gabungan data inner validation. Mengembalikan baris hasil.

In [ ]:
def ensemble_fold(root, members, name, fold, Y_by_id, out):
    preds_dir = root / "results" / "preds_ext"
    member_files = [np.load(preds_dir / f"tf__{member}__f{fold}.npz") for member in members]
    assert all(
        (member_file["inner_ids"] == member_files[0]["inner_ids"]).all()
        and (member_file["test_ids"] == member_files[0]["test_ids"]).all()
        for member_file in member_files
    )
    p_val = np.mean([member_file["inner_prob"] for member_file in member_files], 0)
    p_test = np.mean([member_file["test_prob"] for member_file in member_files], 0)
    y_val = Y_by_id.loc[member_files[0]["inner_ids"]].to_numpy(int)
    y_test = Y_by_id.loc[member_files[0]["test_ids"]].to_numpy(int)
    thresholds = tune_probability_thresholds(y_val, p_val)
    np.savez_compressed(
        preds_dir / f"ens__{name}__f{fold}.npz",
        test_ids=member_files[0]["test_ids"],
        test_prob=p_test.astype(np.float32),
        inner_ids=member_files[0]["inner_ids"],
        inner_prob=p_val.astype(np.float32),
        thresholds=thresholds.astype(np.float32),
    )
    previous = pd.read_csv(out)
    member_rows = previous.config.isin([f"tf__{member}__tuned" for member in members]) & (
        previous.fold == fold
    )
    rows = []
    for tuned, label_thresholds in [(False, np.full(len(LABELS), 0.5)), (True, thresholds)]:
        row = {
            "config": f"ens__{name}" + ("__tuned" if tuned else ""),
            "algo": "ens",
            "strategy": "mo",
            "condition": name,
            "features": "+".join(members),
            "tuned": tuned,
            "fold": fold,
            "n_test": len(y_test),
            "train_time_s": float(previous[member_rows].train_time_s.sum()),
            "inner_macro_f1": round(macro_f1_at(y_val, p_val, thresholds), 4),
            "thresholds": (
                json.dumps([round(float(threshold), 4) for threshold in label_thresholds])
                if tuned
                else ""
            ),
        }
        row.update(evaluate_predictions(y_test, (p_test > label_thresholds[None, :]).astype(int)))
        rows.append(row)
    return rows

## Ensemble Transformer

Membangun ensemble pada setiap fold yang anggota-anggotanya lengkap dan menambahkan baris hasilnya.

In [ ]:
def ensemble_transformers(root, members, name, csv_name=EXTENDED_CSV, log=print):
    data, _ = load_inputs(root)
    Y_by_id = data.set_index("id")[LABELS]
    out = root / "results" / csv_name
    done = completed_runs(out)
    for fold in range(5):
        if (f"ens__{name}__tuned", fold) in done:
            continue
        if not all(
            (root / "results" / "preds_ext" / f"tf__{member}__f{fold}.npz").exists()
            for member in members
        ):
            log(f"ens__{name} f{fold}: members incomplete, skipped")
            continue
        rows = ensemble_fold(root, members, name, fold, Y_by_id, out)
        append_rows(out, rows)
        log(
            f"ens__{name} f{fold}: test macro-F1 tuned={rows[1]['macro_f1']:.4f}, "
            f"inner={rows[1]['inner_macro_f1']:.4f}"
        )

# Strategi Threshold

Threshold per label yang di-tuning pada sekitar 1.500 tweet inner validation dapat overfit, terutama untuk label langka. Empat strategi dibandingkan untuk transformer. Strategi pertama mempertahankan threshold default 0,5. Strategi kedua memakai threshold F1 optimal pada inner validation. Strategi ketiga menggeser threshold hasil tuning setengah jalan ke arah 0,5. Strategi keempat merata-ratakan threshold hasil tuning atas 30 sampel bootstrap dari inner validation.

Setiap strategi diestimasi di dalam inner validation. Threshold dipilih pada satu paruh dan diberi skor pada paruh lainnya, dan pembagian dua bagian ini diulang lima kali. Hanya estimasi ini yang dipakai untuk memilih strategi, dan skor uji dilaporkan sesudahnya.

## Threshold Rata-Rata Bootstrap

Merata-ratakan threshold hasil tuning atas sampel bootstrap dari inner validation.

In [ ]:
def bootstrap_mean_thresholds(Y, probabilities, n_boot=30, seed=0):
    rng = np.random.default_rng(seed)
    n_rows = len(Y)
    samples = (rng.integers(0, n_rows, n_rows) for _ in range(n_boot))
    return np.mean(
        [tune_probability_thresholds(Y[sample], probabilities[sample]) for sample in samples], 0
    )

## Tabel Strategi

Keempat strategi sebagai fungsi dari label dan probabilitas inner validation.

In [ ]:
def fixed_thresholds(Y, probabilities):
    return np.full(Y.shape[1], 0.5)


def shrunk_thresholds(Y, probabilities):
    return (tune_probability_thresholds(Y, probabilities) + 0.5) / 2


THRESHOLD_STRATEGIES = {
    "fixed_0.5": fixed_thresholds,
    "tuned": tune_probability_thresholds,
    "shrunk": shrunk_thresholds,
    "bootstrap_mean": bootstrap_mean_thresholds,
}

## Evaluasi Strategi Threshold

Mengevaluasi setiap strategi pada satu fold. inner_2fold adalah estimasi jujur dari pembagian dua bagian yang diulang di dalam inner validation, dan hanya nilai ini yang boleh dipakai untuk memilih strategi. Skor uji memakai threshold yang dipilih pada seluruh inner validation.

In [ ]:
def evaluate_threshold_strategies(Y_by_id, npz_path, reps=5, seed=0):
    stored = np.load(npz_path)
    y_val, p_val = Y_by_id.loc[stored["inner_ids"]].to_numpy(int), stored["inner_prob"]
    y_test, p_test = Y_by_id.loc[stored["test_ids"]].to_numpy(int), stored["test_prob"]
    rng = np.random.default_rng(seed)
    splits = []
    for _ in range(reps):
        idx = rng.permutation(len(y_val))
        first_half, second_half = idx[: len(idx) // 2], idx[len(idx) // 2 :]
        splits += [(first_half, second_half), (second_half, first_half)]
    out = {}
    for strategy, choose in THRESHOLD_STRATEGIES.items():
        t_full = choose(y_val, p_val)
        inner = np.mean(
            [
                macro_f1_at(
                    y_val[score_half], p_val[score_half], choose(y_val[fit_half], p_val[fit_half])
                )
                for fit_half, score_half in splits
            ]
        )
        out[strategy] = {
            "inner_2fold": inner,
            "test_macro_f1": macro_f1_at(y_test, p_test, t_full),
            "thresholds": t_full,
            "test_ids": stored["test_ids"],
            "test_prob": p_test,
        }
    return out

# Post-Processing dengan Hierarki

Fungsi-fungsi ini menerapkan hierarki anotasi pada skor tersimpan dari sebuah konfigurasi setelah training. Fungsi ini dipakai pada eksperimen lanjutan.

## Aturan Post-Processing

Aturan keputusan biasa dan dua aturan hierarki. Aturan topdown ditetapkan sebagai varian utama sebelum hasil dilihat, dan aturan bottomup dipakai sebagai pembanding.

In [ ]:
POSTPROCESSING_RULES = {
    "original": lambda scores: (scores > 0).astype(int),
    "topdown": lambda scores: enforce_label_hierarchy(scores, "topdown"),
    "bottomup": lambda scores: enforce_label_hierarchy(scores, "bottomup"),
}

## Post-Processing Satu Konfigurasi

Menerapkan setiap aturan pada skor tersimpan dari satu konfigurasi, fold demi fold. Mengembalikan prediksi out-of-fold dan metrik per fold dari setiap aturan.

In [ ]:
def postprocess_config(config, data):
    row_of_id = pd.Series(np.arange(len(data)), index=data["id"])
    Y_all = label_matrix(data)
    oof = {rule: np.zeros_like(Y_all) for rule in POSTPROCESSING_RULES}
    fold_metrics = {rule: [] for rule in POSTPROCESSING_RULES}
    for fold in range(5):
        ids, scores = load_fold_scores(config, fold)
        rows = row_of_id[ids].to_numpy()
        for rule, apply_rule in POSTPROCESSING_RULES.items():
            pred = apply_rule(scores)
            oof[rule][rows] = pred
            fold_metrics[rule].append(evaluate_predictions(Y_all[rows], pred))
    return oof, fold_metrics

## Baris Ringkasan Post-Processing

Meringkas satu konfigurasi: rata-rata metrik setiap aturan dan perubahan yang disebabkan oleh aturan hierarki. Kolom variant berisi penanganan ketidakseimbangan untuk model klasik dan nama konfigurasi untuk transformer.

In [ ]:
def postprocessing_row(config, meta, fold_metrics):
    row = {
        "config": config,
        "algo": meta.algo,
        "strategy": meta.strategy,
        "variant": meta.condition,
        "tuning": meta.tuning,
    }
    f1 = {
        rule: np.mean(
            [[fold_result[f"f1__{label}"] for label in LABELS] for fold_result in metrics], 0
        )
        for rule, metrics in fold_metrics.items()
    }
    for rule, metrics in fold_metrics.items():
        for metric in [
            "macro_f1",
            "minority_macro_f1",
            "jaccard_acc",
            "subset_acc",
            "pred_hierarchy_violation_rate",
        ]:
            row[f"{metric}__{rule}"] = np.mean([fold_result[metric] for fold_result in metrics])
    for rule in ["topdown", "bottomup"]:
        row[f"delta_macro__{rule}"] = row[f"macro_f1__{rule}"] - row["macro_f1__original"]
        row[f"delta_jacc__{rule}"] = row[f"jaccard_acc__{rule}"] - row["jaccard_acc__original"]
        row[f"n_labels_improved__{rule}"] = int((f1[rule] > f1["original"] + 1e-9).sum())
    return row

# Konfigurasi Pembanding

Algoritma dibandingkan melalui konfigurasi pembanding yang ditetapkan sebelum hasil uji dilihat. Setiap konfigurasi pembanding klasik memakai Binary Relevance, fitur kata, tanpa penanganan ketidakseimbangan, dan tuning hiperparameter serta threshold dengan inner validation. Binary Relevance adalah baseline multi-label yang baku, tanpa penanganan ketidakseimbangan menjaga perbandingan tetap tentang algoritmanya, dan tuning yang sama memberi setiap algoritma peluang yang sama. Pembanding transformer adalah IndoBERTweet dengan pengaturan default protokol dan threshold yang di-tuning pada inner validation. Pembanding ini tetap memakai class weighting yang merupakan bagian dari protokol.

## Pengaturan Pembanding

ID konfigurasi setiap pembanding, dengan key berupa nama yang ditampilkan di tabel dan gambar.

In [ ]:
REFERENCE_ALGOS = ["cnb", "svm", "xgb", "mlp"]
REFERENCE_CONFIGS = {
    ALGO_NAMES[algo]: config_id(algo, "br", "none", "tuned") for algo in REFERENCE_ALGOS
}
REFERENCE_CONFIGS["IndoBERTweet"] = "tf__indobertweet_v2__tuned"

## F1 per Label dan Skor per Fold

Mengembalikan rata-rata F1 setiap label dan macro F1 per fold dari satu konfigurasi.

In [ ]:
def per_label_and_folds(runs, config):
    config_runs = runs[runs.config == config]
    return (
        np.array([config_runs[f"f1__{label}"].mean() for label in LABELS]),
        config_runs.sort_values("fold").macro_f1.to_numpy(),
    )

## Warna Pembanding

Warna setiap konfigurasi pembanding pada gambar.

In [ ]:
ALGO_COLORS = {
    "Complement NB": "#7f8c8d",
    "LinearSVC": "#2e86c1",
    "XGBoost": "#c0392b",
    "MLP": "#27ae60",
    "IndoBERTweet": "#8e44ad",
}

## Model Pembanding

Prediksi out-of-fold dan macro F1 per fold dari setiap konfigurasi pembanding yang memiliki kelima fold.

In [ ]:
def reference_models(runs, data):
    models = {}
    for name, config in REFERENCE_CONFIGS.items():
        fold_f1 = per_label_and_folds(runs, config)[1]
        if len(fold_f1) == 5:
            models[name] = (oof_predictions(config, data), fold_f1)
    return models

# Benchmark Inferensi

Biaya sebuah model dalam sistem moderasi terutama bergantung pada seberapa cepat model melabeli teks baru. Fungsi-fungsi ini mengukur waktu prediksi 1.000 tweet di mesin saat ini, termasuk ekstraksi fitur atau tokenisasi. Waktu inferensi tidak bergantung pada bobot hasil training, sehingga waktu transformer diukur dengan bobot pretrained-nya.

## Ukur Waktu Inferensi Model Klasik

Melatih algoritma pembanding dengan hiperparameter tetapnya pada data latih fold 0, lalu mengukur waktu prediksi 1.000 tweet uji. Yang dikembalikan adalah median dari beberapa pengulangan.

In [ ]:
def time_classical_inference(algo, data, folds, n_texts=1000, repeats=5):
    train_rows, test_rows = fold_rows(data, folds[0])
    texts_train, texts_test = (
        data.text_clean.iloc[train_rows],
        data.text_clean.iloc[test_rows[:n_texts]],
    )
    vectorizers = fit_features(texts_train)
    model = fit_multilabel(
        "br",
        algo,
        transform_features(vectorizers, texts_train),
        label_matrix(data)[train_rows],
        False,
    )
    seconds = []
    for _ in range(repeats):
        start_time = time.perf_counter()
        decision_scores(model, transform_features(vectorizers, texts_test))
        seconds.append(time.perf_counter() - start_time)
    return float(np.median(seconds))

## Ukur Waktu Inferensi Transformer

Mengukur waktu tokenisasi dan prediksi teks yang diberikan dengan transformer pretrained. Lintasan pertama dipakai untuk warm-up device dan tidak dihitung.

In [ ]:
def time_transformer_inference(model_name, texts, repeats=3, batch_size=64):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=len(LABELS), problem_type="multi_label_classification"
    ).to(DEVICE)
    seconds = []
    for _ in range(repeats + 1):
        start_time = time.perf_counter()
        enc = encode_texts(tokenizer, texts, 128)
        predict_probabilities(model, enc, np.arange(len(texts)), batch_size, DEVICE)
        seconds.append(time.perf_counter() - start_time)
    del model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return float(np.median(seconds[1:]))

# Fungsi Bantu Lintas Korpus

Fungsi-fungsi ini dipakai pada evaluasi lintas korpus. Anotasi IndoToxic2024 diagregasi dengan majority vote untuk setiap label. Teks dengan vote seimbang dan teks yang ditandai sebagian besar anotator sebagai noise atau spam dihapus. Setiap model mengembalikan decision score, dengan skor positif berarti prediksi positif, sehingga average precision yang tidak bergantung pada threshold dapat dilaporkan di samping F1.

## Pengaturan Lintas Korpus

Pemetaan label dan model yang diberi skor pada korpus 2024. Model klasiknya adalah keempat konfigurasi pembanding. Transformer dan ensemble memakai threshold 0,5, sesuai hasil evaluasi strategi threshold.

In [ ]:
CROSS_CORPUS_FILE = EXTERNAL / "IndoDiscourse__indotoxic2024_annotated_data_v2_final.csv"
CROSS_LABEL_PAIRS = [
    ("HS", "toxicity"),
    ("HS", "identity_attack"),
    ("Abusive", "profanity_obscenity"),
]
CROSS_CLASSICAL = [(algo, "br", "word", "none") for algo in REFERENCE_ALGOS]
CROSS_TRANSFORMERS = {
    "indobertweet_v2": {"model": "indolem/indobertweet-base-uncased"},
    "indobert_p1_v2": {"model": "indobenchmark/indobert-base-p1"},
    "indobertweet_v2_e10": {"model": "indolem/indobertweet-base-uncased", "epochs": 10},
    "indobertweet_v2_e10_s1": {
        "model": "indolem/indobertweet-base-uncased",
        "epochs": 10,
        "seed": 1,
    },
    "indobertweet_v2_e10_s2": {
        "model": "indolem/indobertweet-base-uncased",
        "epochs": 10,
        "seed": 2,
    },
    "indobert_p1_v2_e10": {"model": "indobenchmark/indobert-base-p1", "epochs": 10},
}
FINAL_ENSEMBLE_NAME = "ensemble 3-seed IndoBERTweet + IndoBERT-p1, 10 epochs (final model)"
CROSS_ENSEMBLES = {
    "ensemble IndoBERTweet + IndoBERT-p1 (6 epochs)": ["indobertweet_v2", "indobert_p1_v2"],
    FINAL_ENSEMBLE_NAME: [
        "indobertweet_v2_e10",
        "indobertweet_v2_e10_s1",
        "indobertweet_v2_e10_s2",
        "indobert_p1_v2_e10",
    ],
}

## Majority Vote

Majority vote dari para anotator untuk satu label. Vote yang seimbang menghasilkan nilai kosong.

In [ ]:
def majority_vote(cell):
    votes = [int(vote) for vote in ast.literal_eval(cell)] if isinstance(cell, str) else [int(cell)]
    positives = sum(votes)
    if 2 * positives == len(votes):
        return np.nan
    return int(2 * positives > len(votes))

## Muat IndoToxic2024

Memuat korpus, mengagregasi vote, serta menghapus teks noise atau spam dan teks kosong.

In [ ]:
def load_indotoxic2024(path):
    raw_table = pd.read_csv(path, keep_default_na=False)
    out = pd.DataFrame({"text_id": raw_table.text_id, "text": raw_table.text.astype(str)})
    for col in ["is_noise_or_spam_text", "toxicity", "identity_attack", "profanity_obscenity"]:
        out[col] = raw_table[col].map(majority_vote)
    keep = (out.is_noise_or_spam_text != 1) & (out.text.str.strip().str.len() > 0)
    return out[keep].reset_index(drop=True)

## Interval Bootstrap untuk F1

Interval bootstrap 95 persen untuk F1 kelas positif, dengan resampling teks.

In [ ]:
def bootstrap_interval_f1(labels, pred, n_boot=N_BOOTSTRAP, seed=RANDOM_STATE, chunk=200):
    labels, pred = np.asarray(labels).astype(bool), np.asarray(pred).astype(bool)
    rng = np.random.default_rng(seed)
    values = []
    for start in range(0, n_boot, chunk):
        idx = rng.integers(0, len(labels), (min(chunk, n_boot - start), len(labels)))
        tp, fp, fn = (
            (pred & labels)[idx].sum(1),
            (pred & ~labels)[idx].sum(1),
            (~pred & labels)[idx].sum(1),
        )
        values.append(np.where(2 * tp + fp + fn > 0, 2 * tp / np.maximum(2 * tp + fp + fn, 1), 0))
    return np.percentile(np.concatenate(values), [2.5, 97.5])

## Metrik Biner

F1 kelas positif beserta interval bootstrap-nya, precision, recall, average precision, prevalensi, F1 dari classifier trivial yang selalu memprediksi positif, dan proporsi prediksi positif.

In [ ]:
def binary_metrics(labels, scores):
    labels, scores = np.asarray(labels).astype(int), np.asarray(scores, float)
    pred = (scores > 0).astype(int)
    low, high = bootstrap_interval_f1(labels, pred)
    prevalence = float(labels.mean())
    return {
        "f1_pos": f1_score(labels, pred, zero_division=0),
        "f1_ci95_low": low,
        "f1_ci95_high": high,
        "precision": precision_score(labels, pred, zero_division=0),
        "recall": recall_score(labels, pred, zero_division=0),
        "average_precision": average_precision_score(labels, scores),
        "prevalence": prevalence,
        "f1_always_positive": 2 * prevalence / (1 + prevalence),
        "predicted_positive_rate": float(pred.mean()),
        "n": int(len(labels)),
    }

## Beri Skor Prediksi Lintas Korpus

Memberi skor satu model untuk setiap pasangan label pada pemetaan, hanya memakai teks yang labelnya dinilai.

In [ ]:
def score_cross_corpus(corpus, scores, model_name):
    rows = []
    for source, target in CROSS_LABEL_PAIRS:
        rated = corpus[target].notna().to_numpy()
        y_true = corpus.loc[rated, target].astype(int).to_numpy()
        rows.append(
            {
                "model": model_name,
                "source_label": source,
                "target_label": target,
                **binary_metrics(y_true, scores[rated, LABEL_INDEX[source]]),
            }
        )
    return rows

## Latih Model Klasik pada Seluruh Data

Melatih pada seluruh data Ibrohim dan Budi dengan hiperparameter dan threshold dari inner validation, lalu mengembalikan decision score pada korpus 2024.

In [ ]:
def train_classical_full(
    root, corpus, algo="svm", strategy="br", features="word", condition="none"
):
    data, _ = load_inputs(root)
    Y = label_matrix(data)
    override, thresholds, _ = select_on_inner_validation(
        algo, strategy, condition, data.text_clean.reset_index(drop=True), Y, features
    )
    vectorizers = fit_features(data.text_clean, features)
    model = fit_multilabel(
        strategy,
        algo,
        transform_features(vectorizers, data.text_clean),
        Y,
        condition == "weight",
        override,
    )
    return decision_scores(model, transform_features(vectorizers, corpus.text_clean), thresholds)

## Latih Transformer pada Seluruh Data

Melatih transformer pada seluruh data Ibrohim dan Budi, dengan 15 persen disisihkan untuk pemilihan epoch, dan menyimpan probabilitasnya pada korpus 2024. Hasil yang sudah tersimpan dipakai ulang.

In [ ]:
def train_transformer_full(root, name, cfg, corpus, log=print):
    out = root / "results" / "crosscorpus" / f"tf__{name}.npz"
    if out.exists():
        return np.load(out)["prob"]
    out.parent.mkdir(parents=True, exist_ok=True)
    settings = {**TRANSFORMER_DEFAULTS, **cfg}
    set_global_seed(settings["seed"])
    data, _ = load_inputs(root)
    Y = data[LABELS].to_numpy(dtype=np.float32)
    i_fit, i_val = inner_validation_split(Y.astype(int))
    tokenizer = AutoTokenizer.from_pretrained(settings["model"])
    model = AutoModelForSequenceClassification.from_pretrained(
        settings["model"], num_labels=len(LABELS), problem_type="multi_label_classification"
    ).to(DEVICE)
    texts = list(data["Tweet"].map(normalize_for_transformer)) + list(
        corpus["text"].map(normalize_for_transformer)
    )
    enc = encode_texts(tokenizer, texts, settings["max_len"])
    Y_all = np.vstack(
        [Y, np.zeros((len(corpus), len(LABELS)), np.float32)]
    )  # baris 2024 tidak pernah dipakai untuk pelatihan
    best, _, train_time = fine_tune_with_epoch_selection(
        settings, model, enc, Y_all, i_fit, i_val, DEVICE, log, f"cross-corpus {name}"
    )
    prob = predict_probabilities(
        model, enc, np.arange(len(data), len(texts)), settings["batch"] * 4, DEVICE
    )
    np.savez_compressed(
        out,
        prob=prob.astype(np.float32),
        text_id=corpus.text_id.to_numpy().astype(str),
        best_epoch=best["epoch"],
        train_time_s=train_time,
    )
    del model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return prob

## Analisis Out-of-Vocabulary

Proporsi token dan tipe kata pada korpus 2024 yang tidak pernah muncul di data Ibrohim dan Budi, beserta kata-kata tersebut yang paling sering muncul.

In [ ]:
def oov_analysis(root, corpus, top=30):
    data, _ = load_inputs(root)
    vocabulary = set(" ".join(data.text_clean).split())
    tokens = pd.Series(" ".join(corpus.text_clean).split())
    oov = tokens[~tokens.isin(vocabulary)]
    summary = {
        "oov_token_rate": len(oov) / max(len(tokens), 1),
        "oov_type_rate": oov.nunique() / max(tokens.nunique(), 1),
        "n_token": len(tokens),
    }
    return summary, oov.value_counts().head(top).rename_axis("token").reset_index(name="frequency")

## Siapkan Data Lintas Korpus

Memuat korpus 2024 dan menerapkan preprocessing yang sama dengan model klasik.

In [ ]:
def prepare_cross_corpus(root):
    corpus = load_indotoxic2024(root / "data" / "external" / CROSS_CORPUS_FILE.name)
    preprocess = build_preprocessor(
        load_slang_dictionary(root / "data" / "raw" / "new_kamusalay.csv")
    )
    corpus["text_clean"] = corpus.text.map(preprocess)
    return corpus

## Jalankan Evaluasi Lintas Korpus

Memberi skor pada pembanding klasik, transformer, dan ensemble di korpus 2024. Probabilitas transformer dikurangi 0,5 menjadi decision score-nya.

In [ ]:
def run_cross_corpus(root, run_transformers=True, log=print):
    corpus = prepare_cross_corpus(root)
    rows = []
    for algo, strategy, features, condition in CROSS_CLASSICAL:
        scores = train_classical_full(root, corpus, algo, strategy, features, condition)
        rows += score_cross_corpus(
            corpus, scores, f"{algo}__{strategy}__{condition}__{features}__tuned"
        )
    probs = {}
    for name, cfg in CROSS_TRANSFORMERS.items():
        if run_transformers or (root / "results" / "crosscorpus" / f"tf__{name}.npz").exists():
            probs[name] = train_transformer_full(root, name, cfg, corpus, log)
    for name, prob in probs.items():
        rows += score_cross_corpus(corpus, prob - 0.5, f"tf__{name} (threshold 0.5)")
    for ensemble_name, members in CROSS_ENSEMBLES.items():
        if all(member in probs for member in members):
            rows += score_cross_corpus(
                corpus, np.mean([probs[member] for member in members], 0) - 0.5, ensemble_name
            )
    oov_summary, oov_top = oov_analysis(root, corpus)
    return corpus, pd.DataFrame(rows), oov_summary, oov_top

## Konfigurasi In-Domain

Konfigurasi di log run yang sesuai dengan setiap model pada evaluasi lintas korpus, dipakai untuk mencari F1 in-domain-nya.

In [ ]:
IN_DOMAIN_CONFIG = {
    **{
        f"{algo}__{strategy}__{condition}__{features}__tuned": config_id(
            algo, strategy, condition, "tuned", features
        )
        for algo, strategy, features, condition in CROSS_CLASSICAL
    },
    **{f"tf__{name} (threshold 0.5)": f"tf__{name}" for name in CROSS_TRANSFORMERS},
    "ensemble IndoBERTweet + IndoBERT-p1 (6 epochs)": "ens__indobertweet+indobert_p1",
    FINAL_ENSEMBLE_NAME: "ens__e10_indobertweet_x3+indobert_p1",
}

## F1 In-Domain

Rata-rata F1 cross-validation dari satu label untuk satu model lintas korpus, atau NaN jika model tersebut tidak memiliki run.

In [ ]:
def in_domain_f1(runs, model_name, label):
    model_runs = runs[runs.config == IN_DOMAIN_CONFIG.get(model_name)]
    return model_runs[f"f1__{label}"].mean() if len(model_runs) else np.nan

# Sanity Check

Pemeriksaan kecil ini menggantikan unit test dan dijalankan sebelum eksperimen apa pun. Cell terakhir mendaftar pemeriksaan yang lolos.

## Pemeriksaan TF-IDF Deterministik

Frekuensi yang sama dipecah secara alfabetis, apa pun urutan dokumennya.

In [ ]:
sanity_checks = []
docs = ["bb aa cc", "cc bb aa", "dd ee", "ee dd", "zz yy"]
check_params = {"ngram_range": (1, 1), "max_features": 3, "sublinear_tf": True, "analyzer": "word"}
vocab_a = fit_deterministic_tfidf(docs, **check_params).get_feature_names_out()
vocab_b = fit_deterministic_tfidf(docs[::-1], **check_params).get_feature_names_out()
assert list(vocab_a) == ["aa", "bb", "cc"] == list(vocab_b)
sanity_checks.append("deterministic TF-IDF breaks ties alphabetically")

## Pemeriksaan Hierarki

Repair dan post-processing sama-sama menghasilkan label set yang mengikuti aturan anotasi.

In [ ]:
y_check = np.zeros(len(LABELS), dtype=int)
y_check[LABEL_INDEX["HS_Gender"]] = 1
s_check = np.zeros(len(LABELS))
s_check[LABEL_INDEX["HS_Group"]], s_check[LABEL_INDEX["HS_Strong"]] = 2, 3
repaired_check = repair_synthetic_labelset(y_check, s_check, np.zeros(len(LABELS)))
assert count_hierarchy_violations(repaired_check[None, :])["any_violation_rows"] == 0
S_check = np.random.default_rng(1).normal(size=(500, len(LABELS)))
for mode in ("topdown", "bottomup"):
    assert (
        count_hierarchy_violations(enforce_label_hierarchy(S_check, mode))["any_violation_rows"]
        == 0
    )
sanity_checks.append("hierarchy repair and post-processing follow the annotation rules")

## Pemeriksaan Threshold

Threshold hasil tuning tidak pernah lebih buruk daripada threshold default pada data yang sama, dan pencarian threshold cepat memberikan jawaban yang sama dengan loop biasa atas kandidat yang sama.

In [ ]:
def reference_threshold(labels, scores):
    if labels.sum() == 0:
        return 0.0
    candidates = np.unique(
        np.concatenate([np.quantile(scores, np.linspace(0.01, 0.99, 99)), [0.0]])
    )
    best_f, best_t = -1.0, 0.0
    for threshold in candidates:
        f1_value = f1_score(labels, (scores > threshold).astype(int), zero_division=0)
        if f1_value > best_f + 1e-12 or (
            abs(f1_value - best_f) <= 1e-12 and abs(threshold) < abs(best_t)
        ):
            best_f, best_t = f1_value, float(threshold)
    return best_t


rng_check = np.random.default_rng(0)
y_bin = (rng_check.random(500) < 0.1).astype(int)
s_bin = y_bin * 0.3 + rng_check.normal(0, 0.2, 500) - 0.4
tuned_f1 = f1_score(y_bin, (s_bin > f1_optimal_threshold(y_bin, s_bin)).astype(int))
assert tuned_f1 >= f1_score(y_bin, (s_bin > 0).astype(int))
for _ in range(100):
    y_rand = (rng_check.random(300) < rng_check.uniform(0.02, 0.5)).astype(int)
    s_rand = y_rand * rng_check.uniform(0, 1) + rng_check.normal(0, 0.3, 300)
    assert f1_optimal_threshold(y_rand, s_rand) == reference_threshold(y_rand, s_rand)
sanity_checks.append("tuned thresholds and the fast threshold search")

## Pemeriksaan Metrik dan Statistik

Fungsi macro F1 cepat sama dengan nilai scikit-learn, permutation test memberikan p sama dengan 1 untuk prediksi yang identik, serta interval dan equivalence test berperilaku sesuai harapan.

In [ ]:
Y_rand = (rng_check.random((400, len(LABELS))) < 0.2).astype(int)
P_rand = rng_check.random((400, len(LABELS)))
pred_rand = (P_rand > 0.5).astype(int)
sklearn_macro = f1_score(Y_rand, pred_rand, average="macro", zero_division=0)
assert abs(macro_f1_at(Y_rand, P_rand, 0.5) - sklearn_macro) < 1e-12
assert (
    abs(macro_f1_from_counts(count_matrices(Y_rand, pred_rand).sum(axis=0)) - sklearn_macro) < 1e-6
)
assert permutation_test_macro_f1(Y_rand, pred_rand, pred_rand, n_permutations=200)[1] == 1.0
np.testing.assert_allclose(holm_bonferroni([0.01, 0.04, 0.03]), [0.03, 0.06, 0.06])
interval_check = corrected_t_interval([0.001, 0.002, 0.0, 0.001, 0.002], 10000, 2500)
assert (
    interval_check["low"] < 0.0012 < interval_check["high"]
    and interval_check["p_equivalence"] < 0.05
)
sanity_checks.append("fast macro F1, permutation test, Holm correction and equivalence test")
pd.DataFrame({"check": sanity_checks, "passed": True})

# Analisis Data Eksploratif

Semua statistik di bagian ini menggambarkan data mentah sebanyak 13.169 baris sebelum deduplikasi. Statistik data akhir ada pada langkah deduplikasi.

## Data Mentah

Ukuran data mentah dan baris-baris awalnya, dengan kata kasar disamarkan.

### Muat Data Mentah

Membaca tweet, kamus slang, dan leksikon kata kasar.

In [ ]:
raw = load_raw_dataset(RAW / "re_dataset.csv")
slang_dict = load_slang_dictionary(RAW / "new_kamusalay.csv")
abusive_lexicon = load_abusive_lexicon(RAW / "abusive.csv")
raw_summary = pd.DataFrame(
    [
        {
            "raw_rows": len(raw),
            "columns": raw.shape[1],
            "slang_dictionary_entries": len(slang_dict),
            "abusive_lexicon_words": len(abusive_lexicon),
        }
    ]
)

### Ukuran Data

In [ ]:
raw_summary

### Baris Awal

In [ ]:
raw.assign(Tweet=raw.Tweet.map(lambda tweet: mask_abusive_words(tweet, abusive_lexicon))).head()

## Frekuensi Label dan Imbalance Ratio

Label yang muncul pada kurang dari 5 persen tweet ditandai sebagai label minoritas. IRLbl dan MeanIR mengikuti Charte et al. (2015) dan menentukan label mana yang diproses MLSMOTE.

### Hitung Label

Jumlah, proporsi, dan imbalance ratio setiap label, diurutkan berdasarkan jumlah.

In [ ]:
Y_raw = label_matrix(raw)
MEAN_IR_RAW = mean_ir(Y_raw)
label_counts = pd.DataFrame({"label": LABELS, "count": Y_raw.sum(axis=0)})
label_counts["percent"] = (label_counts["count"] / len(raw) * 100).round(2)
label_counts["minority_lt_5pct"] = label_counts["percent"] < 5
label_counts["IRLbl"] = irlbl(Y_raw).round(2)
label_counts["IRLbl_gt_MeanIR"] = label_counts["IRLbl"] > MEAN_IR_RAW
label_counts = label_counts.sort_values("count", ascending=False).reset_index(drop=True)

### MeanIR Data Mentah

In [ ]:
pd.DataFrame({"MeanIR (raw data)": [round(MEAN_IR_RAW, 2)]})

### Tabel Frekuensi Label

In [ ]:
show_table(label_counts, decimals=2)

### Grafik Distribusi Label

Label minoritas digambar dengan warna merah.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
colors = [
    "#c0392b" if is_minority else "#5d6d7e" for is_minority in label_counts["minority_lt_5pct"]
]
ax.bar(label_counts["label"], label_counts["count"], color=colors)
for position, (count, percent) in enumerate(zip(label_counts["count"], label_counts["percent"])):
    ax.text(position, count + 60, f"{percent:.1f}%", ha="center", fontsize=8)
ax.set_ylabel("Number of tweets")
ax.set_title("Label distribution (red = minority label, < 5%)")
plt.xticks(rotation=45, ha="right")
fig.savefig(FIG / "eda_label_distribution.png")
plt.show()

## Label Cardinality dan Label Set

Label cardinality adalah rata-rata jumlah label per tweet, dan label density adalah label cardinality dibagi jumlah label.

### Hitung Label per Tweet

In [ ]:
labels_per_tweet = Y_raw.sum(axis=1)
cardinality = labels_per_tweet.mean()
labelsets = pd.Series(["".join(map(str, row)) for row in Y_raw])
cardinality_summary = pd.DataFrame(
    {
        "metric": [
            "n_tweets",
            "label_cardinality",
            "label_density",
            "n_unique_labelsets",
            "n_labelsets_seen_once",
        ],
        "value": [
            len(raw),
            round(cardinality, 3),
            round(cardinality / Y_raw.shape[1], 3),
            labelsets.nunique(),
            int((labelsets.value_counts() == 1).sum()),
        ],
    }
)
tweets_by_n_labels = (
    pd.Series(labels_per_tweet)
    .value_counts()
    .sort_index()
    .rename_axis("n_labels")
    .reset_index(name="n_tweets")
)

### Ringkasan Label Cardinality

In [ ]:
cardinality_summary

### Tweet Menurut Jumlah Label

In [ ]:
tweets_by_n_labels

## Konsistensi Hierarki

Baris yang melanggar setiap aturan anotasi, dan jumlah kategori pada setiap tweet ujaran kebencian.

### Periksa Aturan Anotasi

In [ ]:
raw_violations = pd.Series(count_hierarchy_violations(Y_raw)).to_frame("n_rows")
categories_per_hs_tweet = (
    raw[raw.HS == 1][CATEGORY_LABELS]
    .sum(axis=1)
    .value_counts()
    .sort_index()
    .rename_axis("n_categories")
    .to_frame("n_hate_speech_tweets")
)

### Pelanggaran Aturan

In [ ]:
raw_violations

### Kategori per Tweet Ujaran Kebencian

In [ ]:
categories_per_hs_tweet

# Preprocessing dan Deduplikasi

Preprocessing untuk model klasik diterapkan tanpa stemming dan tanpa penghapusan stopword, lalu duplikat dihapus.

## Terapkan Preprocessing

Setiap tweet dibersihkan dan dinormalisasi. Lima contoh ditampilkan dengan kata kasar disamarkan.

### Bersihkan dan Normalisasi

In [ ]:
preprocess = build_preprocessor(slang_dict)
raw["text_clean"] = raw.Tweet.map(preprocess)
example_idx = raw.sample(5, random_state=RANDOM_STATE).index
preprocessing_examples = pd.DataFrame(
    {
        "original": raw.loc[example_idx, "Tweet"],
        "cleaned": raw.loc[example_idx, "Tweet"].map(clean_text),
        "normalized": raw.loc[example_idx, "text_clean"],
    }
).apply(lambda column: column.map(lambda text: mask_abusive_words(text, abusive_lexicon)))

### Contoh Preprocessing

In [ ]:
preprocessing_examples

## Deduplikasi dan Simpan

Duplikat dihapus dan data akhir disimpan ke data/processed/data_clean.csv, yang dibaca oleh runner eksperimen. Nilai rows_final adalah ukuran dataset yang dilaporkan di paper.

### Hapus Duplikat dan Simpan

In [ ]:
data, dedup_report = deduplicate(raw[["Tweet", "text_clean"] + LABELS])
data.insert(0, "id", np.arange(len(data)))
assert count_hierarchy_violations(label_matrix(data))["any_violation_rows"] == 0
data.to_csv(PROC / "data_clean.csv", index=False)

### Deskripsikan Data Akhir

Frekuensi label, imbalance ratio, dan label cardinality dari data setelah deduplikasi.

In [ ]:
Y_final = label_matrix(data)
MEAN_IR_FINAL = mean_ir(Y_final)
final_counts = pd.DataFrame({"label": LABELS, "count": Y_final.sum(axis=0)})
final_counts["percent"] = (final_counts["count"] / len(data) * 100).round(2)
final_counts["IRLbl"] = irlbl(Y_final).round(2)
final_counts["IRLbl_gt_MeanIR"] = final_counts["IRLbl"] > MEAN_IR_FINAL
final_data_summary = pd.DataFrame(
    {
        "MeanIR (final data)": [round(MEAN_IR_FINAL, 2)],
        "label_cardinality": [round(Y_final.sum(axis=1).mean(), 3)],
    }
)

### Laporan Deduplikasi

In [ ]:
pd.Series(dedup_report).to_frame("value")

### Ringkasan Data Akhir

In [ ]:
final_data_summary

### Frekuensi Label Akhir

In [ ]:
show_table(final_counts, decimals=2)

# Fold Cross-Validation

Fold dibuat dengan MultilabelStratifiedKFold, yaitu iterative stratification dari Sechidis et al. (2011), dengan lima fold, shuffling, dan random state 42. File results/folds.json menyimpan id baris, bukan posisi, sehingga fold tetap valid jika urutan file data berubah.

## Buat atau Muat Fold

Jika folds.json sudah ada, file itu dimuat dan tidak ada fold baru yang dibuat. Hapus file tersebut hanya setelah tim setuju.

### Buat atau Muat File Fold

In [ ]:
folds_created = not FOLDS_PATH.exists()
if folds_created:
    ids = data["id"].to_numpy()
    splitter = MultilabelStratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    folds = [
        {
            "fold": fold_number,
            "train_ids": ids[train_index].tolist(),
            "test_ids": ids[test_index].tolist(),
        }
        for fold_number, (train_index, test_index) in enumerate(
            splitter.split(np.zeros(len(data)), label_matrix(data))
        )
    ]
    FOLDS_PATH.write_text(json.dumps(folds))
else:
    folds = json.loads(FOLDS_PATH.read_text())

### Periksa Fold

Setiap id harus berada di tepat satu test fold, dan id latih serta id uji tidak pernah tumpang tindih. Data dan fold kemudian dibaca ulang persis seperti yang dilakukan runner eksperimen.

In [ ]:
all_test_ids = np.concatenate([fold["test_ids"] for fold in folds])
assert len(all_test_ids) == len(set(all_test_ids)) == len(data)
for fold in folds:
    assert not set(fold["train_ids"]) & set(fold["test_ids"])
    assert len(fold["train_ids"]) + len(fold["test_ids"]) == len(data)
data, folds = load_inputs(ROOT)
Y = label_matrix(data)
fold_sizes = pd.DataFrame(
    [
        {
            "fold": fold["fold"],
            "train_size": len(fold["train_ids"]),
            "test_size": len(fold["test_ids"]),
            "folds_file": "created" if folds_created else "loaded",
        }
        for fold in folds
    ]
)

### Ukuran Fold

In [ ]:
fold_sizes

## Distribusi Label per Fold

Jumlah setiap label di setiap test fold, dan seberapa besar proporsi setiap label bervariasi antarfold dalam poin persentase.

### Hitung Label per Fold

In [ ]:
fold_label_rows = []
for fold in folds:
    _, test_rows = fold_rows(data, fold)
    fold_label_rows.append(
        {
            "fold": fold["fold"],
            "n_test": len(test_rows),
            **{
                label: int(Y[test_rows, label_index].sum())
                for label_index, label in enumerate(LABELS)
            },
        }
    )
fold_label_dist = pd.DataFrame(fold_label_rows).set_index("fold")
label_share = fold_label_dist[LABELS].div(fold_label_dist.n_test, axis=0) * 100
label_share_range = (label_share.max() - label_share.min()).round(2).to_frame("range_pp").T

### Jumlah Label per Fold

In [ ]:
fold_label_dist

### Rentang Proporsi Label

In [ ]:
label_share_range

# Inspeksi MLSMOTE

MLSMOTE asli dibandingkan dengan MLSMOTE plus hierarchy repair pada data latih fold 0. Fitur dan MLSMOTE hanya dilatih pada data latih tersebut.

## Resampling Fold 0

Kedua versi MLSMOTE dijalankan pada data latih fold 0.

### Bangun Fitur Fold 0

In [ ]:
train_rows_f0, _ = fold_rows(data, folds[0])
texts_f0 = data.text_clean.iloc[train_rows_f0]
X_train_f0 = transform_features(fit_features(texts_f0), texts_f0)
Y_train_f0 = Y[train_rows_f0]

### Jalankan Kedua Versi

Menjalankan MLSMOTE tanpa dan dengan repair, lalu meringkas synthetic instance dari setiap versi.

In [ ]:
mlsmote_runs = {}
for name, repair in [("mlsmote_original", False), ("mlsmote_repair", True)]:
    start_time = time.time()
    _, Y_resampled, summary = mlsmote(X_train_f0, Y_train_f0, n_neighbours=5, repair=repair)
    mlsmote_runs[name] = {"seconds": time.time() - start_time, "summary": summary, "Y": Y_resampled}
fold0_summary = pd.DataFrame(
    [
        {
            "training_rows": X_train_f0.shape[0],
            "features": X_train_f0.shape[1],
            "MeanIR (fold 0 training data)": round(
                mlsmote_runs["mlsmote_original"]["summary"]["mean_ir"], 2
            ),
        }
    ]
)
mlsmote_summary = pd.DataFrame(
    {
        name: {
            "seconds": round(run["seconds"], 2),
            "n_train_before": len(train_rows_f0),
            "n_synthetic": run["summary"]["n_synthetic"],
            "n_repaired": run["summary"]["n_repaired"],
            "n_synthetic_violating_hierarchy": count_hierarchy_violations(
                run["Y"][len(train_rows_f0) :]
            )["any_violation_rows"],
            "processed_labels": ", ".join(
                LABELS[label_index] for label_index in run["summary"]["processed_labels"]
            ),
        }
        for name, run in mlsmote_runs.items()
    }
).T

### Data Latih Fold 0

In [ ]:
fold0_summary

### Ringkasan MLSMOTE

In [ ]:
mlsmote_summary

## Jumlah Label Sebelum dan Sesudah

Jumlah label dan imbalance ratio sebelum dan sesudah kedua versi MLSMOTE.

### Hitung Label

In [ ]:
original_run, repair_run = mlsmote_runs["mlsmote_original"], mlsmote_runs["mlsmote_repair"]
mlsmote_label_counts = pd.DataFrame(
    [
        {
            "label": label,
            "count_before": int(Y_train_f0[:, label_index].sum()),
            "count_original": int(original_run["Y"][:, label_index].sum()),
            "count_repair": int(repair_run["Y"][:, label_index].sum()),
            "IRLbl_before": round(original_run["summary"]["irlbl_before"][label_index], 2),
            "IRLbl_original": round(original_run["summary"]["irlbl_after"][label_index], 2),
            "IRLbl_repair": round(repair_run["summary"]["irlbl_after"][label_index], 2),
        }
        for label_index, label in enumerate(LABELS)
    ]
)

### Jumlah Label

In [ ]:
mlsmote_label_counts

### Grafik Label Minoritas

Label minoritas sebelum dan sesudah MLSMOTE dengan repair.

In [ ]:
to_plot = mlsmote_label_counts.set_index("label").loc[
    MINORITY_LABELS, ["count_before", "count_repair"]
]
to_plot.columns = ["before MLSMOTE", "after MLSMOTE + repair"]
ax = to_plot.plot.bar(figsize=(7, 3.5), color=["#5d6d7e", "#c0392b"])
ax.set_title("Minority labels before and after MLSMOTE (fold 0, training data)")
ax.set_ylabel("Number of instances")
plt.xticks(rotation=0)
plt.savefig(FIG / "mlsmote_fold0_before_after.png")
plt.show()

## Keputusan Desain

Satu kali lintasan MLSMOTE, seperti pada paper, menambahkan satu synthetic instance untuk setiap instance minoritas per label. Imbalance ratio turun, tetapi label seperti HS_Gender tetap menjadi label minoritas. Ini adalah perilaku algoritma aslinya, bukan bug.

Pada versi asli, sebagian synthetic instance melanggar hierarki label, seperti terlihat pada kolom n_synthetic_violating_hierarchy. Karena itu grid utama memakai versi dengan repair, dan versi asli dievaluasi pada ablation repair. Klaim tentang MLSMOTE di paper merujuk pada MLSMOTE dengan hierarchy repair, dan pada algoritma aslinya hanya melalui ablation tersebut.

# Grid Model Klasik

Grid ini menyilangkan empat algoritma, yaitu Complement Naive Bayes, LinearSVC, XGBoost, dan MLP, dengan dua strategi multi-label, yaitu Binary Relevance dan Classifier Chains. Label Powerset hanya dijalankan untuk perbandingan dengan baseline, memakai Complement Naive Bayes, LinearSVC, dan random forest.

Penanganan ketidakseimbangan yang dipakai adalah tanpa penanganan, class weighting, dan MLSMOTE dengan hierarchy repair. MLSMOTE asli tanpa repair dijalankan sebagai ablation. Setiap konfigurasi dijalankan dalam mode fixed dan full tuning pada lima fold iterative stratification dengan seed 42. Complement Naive Bayes dan LinearSVC juga dijalankan dalam mode threshold saja, yang memisahkan threshold tuning dari pencarian hiperparameter. Totalnya 325 run. XGBoost dan MLP hanya dijalankan jika RUN_GPU_MODELS bernilai true, dan jika tidak, hasil tersimpannya yang dipakai.

## Susun Grid Eksperimen

Run eksperimen utama, hiperparameter tetap, dan kandidat tuning.

### Grid Utama

In [ ]:
CLASSICAL_ALGOS = ["cnb", "svm"] + (["xgb", "mlp"] if RUN_GPU_MODELS else [])
main_grid = build_grid(
    CLASSICAL_ALGOS, ["br", "cc"], ["none", "weight", "mlsmote"], tuning=("fixed", "tuned")
)
main_grid += build_grid(
    [algo for algo in CLASSICAL_ALGOS if algo in THRESHOLD_ONLY_ALGOS],
    ["br", "cc"],
    ["none", "weight", "mlsmote"],
    tuning=("thr",),
)
main_grid += build_grid(["cnb", "svm", "rf"], ["lp"], ["none"])
main_grid += build_grid(["cnb", "svm"], ["br"], ["mlsmote_raw"])

### Run per Algoritma dan Mode Tuning

In [ ]:
(
    pd.DataFrame(main_grid, columns=["algo", "strategy", "condition", "tuning", "features", "fold"])
    .groupby(["algo", "tuning"])
    .size()
    .unstack(fill_value=0)
)

### Hiperparameter Tetap

In [ ]:
pd.DataFrame({algo: [params] for algo, params in BASE_PARAMS.items()}).T.rename(
    columns={0: "fixed hyperparameters"}
)

### Kandidat Tuning

In [ ]:
pd.Series(HYPERPARAMETER_GRID, name="tuning candidates").to_frame()

## Periksa Stale Run

Menghitung stale run serta run XGBoost dan MLP yang dibuat di device lain pada kedua log run. Stale run dihapus jika REMOVE_STALE_RUNS bernilai true, dan run dari device lain dihapus jika REMOVE_OTHER_DEVICE_RUNS bernilai true, sehingga grid menjalankannya lagi di sini. Setiap run XGBoost dan MLP mencatat nama GPU-nya, sehingga run yang dibuat di mesin ini tidak pernah dihapus dan notebook aman dijalankan ulang setelah terhenti. Tidak ada run XGBoost dan MLP yang dihapus jika keduanya tidak dijalankan di mesin ini.

### Hapus Run Lama

In [ ]:
removal_rows = []
for csv_name, preds_subdir in [(RESULTS_CSV, "preds"), (EXTENDED_CSV, "preds_ext")]:
    path = RES / csv_name
    if not path.exists():
        continue
    runs = pd.read_csv(path)
    stale = runs.index.isin(find_stale_runs(path).index)
    other_device = runs.algo.isin(GPU_ALGOS) & (runs.device != DEVICE_LABEL)
    remove = (REMOVE_STALE_RUNS & stale) | (REMOVE_OTHER_DEVICE_RUNS & other_device)
    if not RUN_GPU_MODELS:
        remove &= ~runs.algo.isin(GPU_ALGOS)
    if remove.any():
        remove_runs(path, runs[remove], preds_subdir)
    removal_rows.append(
        {
            "run_log": csv_name,
            "stale_runs": int(stale.sum()),
            "gpu_runs_from_other_devices": int(other_device.sum()),
            "removed": int(remove.sum()),
        }
    )
removal_summary = pd.DataFrame(removal_rows)

### Run yang Dihapus

In [ ]:
removal_summary

## Jalankan Grid Eksperimen

Run yang sudah ada di experiments.csv dilewati. Ini adalah langkah terlama di notebook.

### Jalankan Grid Utama

In [ ]:
if RUN_GRID:
    run_grid(
        ROOT,
        main_grid,
        n_jobs_cpu=N_JOBS_CPU,
        csv_name=RESULTS_CSV,
        preds_subdir="preds",
        verbose=False,
    )
main_log = pd.read_csv(RES / RESULTS_CSV)

### Run di Log

In [ ]:
main_log.groupby("algo").agg(configurations=("config", "nunique"), runs=("config", "size"))

# Evaluasi dan Uji Statistik

Setiap perbandingan memakai paired permutation test pada prediksi out-of-fold dan corrected resampled t-test pada lima fold, lengkap dengan interval 95 persen untuk selisih macro F1 dan equivalence test terhadap margin yang ditetapkan di konfigurasi. Verdict dijelaskan pada bagian Uji Statistik. Hasil per label adalah deskripsi, bukan unit statistik.

## Muat Hasil

Memuat run lengkap dari grid utama tanpa baris protokol IndoBERTweet sebelumnya, serta gabungan kedua log run untuk konfigurasi pembanding. Corrected resampled t-test memakai ukuran outer training fold dan outer test fold.

### Pengaturan Evaluasi

Kolom yang mengidentifikasi sebuah konfigurasi, penanganan ketidakseimbangan pada grid utama, dan metrik tabel utama.

In [ ]:
KEY = ["algo", "strategy", "condition", "tuning"]
CONDITIONS_MAIN = ["none", "weight", "mlsmote"]
MAIN_METRICS = [
    "macro_f1",
    "micro_f1",
    "minority_macro_f1",
    "jaccard_acc",
    "hamming_loss",
    "pred_hierarchy_violation_rate",
    "total_time_s",
]

### Muat Run

In [ ]:
results = load_complete_runs(RES / RESULTS_CSV)
results = results[results.algo != "bert"]
all_runs = load_all_runs()
ALGOS = [algo for algo in ["cnb", "svm", "xgb", "mlp"] if algo in results.algo.unique()]
N_TRAIN = int(np.mean([len(fold["train_ids"]) for fold in folds]))
N_TEST = int(np.mean([len(fold["test_ids"]) for fold in folds]))

### Run yang Dimuat

In [ ]:
pd.DataFrame(
    [
        {
            "configurations": results.config.nunique(),
            "runs": len(results),
            "outer_train_size": N_TRAIN,
            "outer_test_size": N_TEST,
        }
    ]
)

## Tabel Utama

Rata-rata dan simpangan baku atas lima fold untuk setiap konfigurasi grid utama, diurutkan berdasarkan macro F1.

### Hitung Rata-Rata dan Simpangan Baku

In [ ]:
grouped = results.groupby(KEY)[MAIN_METRICS]
config_mean, config_std = grouped.mean(), grouped.std()
config_mean_flat = config_mean.reset_index()
main_table = (
    config_mean.round(4).astype(str) + " ± " + config_std.round(4).astype(str)
).reset_index()
main_table["algo"] = main_table.algo.map(ALGO_NAMES)
main_table = main_table.loc[config_mean_flat.macro_f1.sort_values(ascending=False).index]

### Konfigurasi Diurutkan Berdasarkan Macro F1

In [ ]:
show_table(main_table, decimals=None)

## Grafik Penanganan Ketidakseimbangan

Macro F1 dan macro F1 minoritas untuk setiap penanganan ketidakseimbangan pada Binary Relevance, dalam ketiga mode tuning.

### Panel Grafik

Metrik untuk setiap baris panel beserta judulnya, dan mode tuning untuk setiap kolom.

In [ ]:
IMBALANCE_CHART_METRICS = {"macro_f1": "Macro-F1", "minority_macro_f1": "Minority macro-F1"}
IMBALANCE_CHART_TUNINGS = ["fixed", "thr", "tuned"]

### Siapkan Panel

Satu tabel algoritma terhadap penanganan ketidakseimbangan untuk setiap metrik dan mode tuning. Panel tanpa run lengkap tidak disertakan.

In [ ]:
imbalance_panels = {}
for metric, tuning in product(IMBALANCE_CHART_METRICS, IMBALANCE_CHART_TUNINGS):
    subset = config_mean_flat[
        (config_mean_flat.strategy == "br")
        & (config_mean_flat.tuning == tuning)
        & config_mean_flat.condition.isin(CONDITIONS_MAIN)
    ]
    if subset.empty:
        continue
    pivot = subset.pivot(index="algo", columns="condition", values=metric).reindex(
        index=ALGOS, columns=CONDITIONS_MAIN
    )
    pivot.index = pivot.index.map(ALGO_NAMES)
    pivot.columns = [IMBALANCE_NAMES[condition] for condition in pivot.columns]
    imbalance_panels[(metric, tuning)] = pivot

### Grafik Batang per Mode Tuning

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 7), sharey="row")
for (row, metric), (column, tuning) in product(
    enumerate(IMBALANCE_CHART_METRICS), enumerate(IMBALANCE_CHART_TUNINGS)
):
    ax = axes[row, column]
    if (metric, tuning) not in imbalance_panels:
        ax.set_visible(False)
        continue
    imbalance_panels[(metric, tuning)].plot.bar(
        ax=ax, color=["#95a5a6", "#2e86c1", "#c0392b"], width=0.75, legend=False
    )
    ax.set_title(f"{IMBALANCE_CHART_METRICS[metric]} | BR | {tuning}", fontsize=10)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=0)
handles, legend_labels = axes[0, 0].get_legend_handles_labels()
fig.legend(
    handles,
    legend_labels,
    title="imbalance treatment",
    loc="upper center",
    ncol=3,
    bbox_to_anchor=(0.5, 1.09),
)
fig.tight_layout()
fig.savefig(FIG / "macro_f1_by_condition.png")
plt.show()

## RQ2 Penanganan Ketidakseimbangan dan Tuning

Perbandingan dipasangkan di dalam algoritma dan strategi yang sama. Family imbalance membandingkan none dengan weight, none dengan mlsmote, dan weight dengan mlsmote, semuanya dalam mode fixed. Family threshold tuning membandingkan fixed dengan thr, family hyperparameter tuning membandingkan thr dengan tuned, dan family full tuning membandingkan fixed dengan tuned, untuk setiap penanganan ketidakseimbangan. Family threshold tuning dan hyperparameter tuning ada untuk Complement Naive Bayes dan LinearSVC, yaitu algoritma yang dijalankan dalam mode threshold saja.

### Pasangan Perbandingan

Setiap entri RQ2_SETTINGS memasangkan pengaturan A dengan pengaturan B, berupa penanganan ketidakseimbangan dan mode tuning, serta menamai family-nya. Pasangan mengulang setiap entri untuk setiap algoritma dan strategi.

In [ ]:
RQ2_SETTINGS = (
    [
        ((treatment_a, "fixed"), (treatment_b, "fixed"), "imbalance")
        for treatment_a, treatment_b in [
            ("none", "weight"),
            ("none", "mlsmote"),
            ("weight", "mlsmote"),
        ]
    ]
    + [
        ((condition, "fixed"), (condition, "thr"), "threshold tuning")
        for condition in CONDITIONS_MAIN
    ]
    + [
        ((condition, "thr"), (condition, "tuned"), "hyperparameter tuning")
        for condition in CONDITIONS_MAIN
    ]
    + [((condition, "fixed"), (condition, "tuned"), "full tuning") for condition in CONDITIONS_MAIN]
)
rq2_pairs = [
    {
        "family": family,
        "algo": ALGO_NAMES[algo],
        "strategy": strategy.upper(),
        "A": f"{condition_a}/{tuning_a}",
        "B": f"{condition_b}/{tuning_b}",
        "config_a": config_id(algo, strategy, condition_a, tuning_a),
        "config_b": config_id(algo, strategy, condition_b, tuning_b),
    }
    for algo, strategy, ((condition_a, tuning_a), (condition_b, tuning_b), family) in product(
        ALGOS, ["br", "cc"], RQ2_SETTINGS
    )
]

### Jalankan Perbandingan

Pasangan yang konfigurasinya belum lengkap dilewati.

In [ ]:
rq2 = add_verdicts(compare_configs(rq2_pairs, results, data, N_TRAIN, N_TEST), by="family")

### Verdict per Family

In [ ]:
pd.crosstab(rq2.family, rq2.verdict)

### Semua Perbandingan

In [ ]:
show_table(rq2.drop(columns=["config_a", "config_b"]))

### Grafik Perbandingan

Setiap perbandingan ditampilkan sebagai selisih macro F1 dengan interval 95 persennya, satu panel per family.

In [ ]:
families = list(rq2.family.unique())
fig, axes = plt.subplots(
    1,
    len(families),
    figsize=(5.5 * len(families), 0.3 * rq2.family.value_counts().max() + 2),
    squeeze=False,
)
for ax, family in zip(axes[0], families):
    subset = rq2[rq2.family == family].assign(
        label=lambda table: table.algo + " " + table.strategy + " | " + table.A + " vs " + table.B
    )
    plot_comparisons(subset, "label", family, ax)
handles = [
    plt.Line2D([], [], marker="o", ls="", color=color, label=verdict)
    for verdict, color in VERDICT_COLORS.items()
]
fig.legend(handles=handles, loc="upper center", ncol=4, bbox_to_anchor=(0.5, 1.04))
fig.tight_layout()
fig.savefig(FIG / "rq2_comparisons.png")
plt.show()

## RQ3 Algoritma di Berbagai Pengaturan

Rata-rata macro F1 keempat algoritma klasik pada setiap pengaturan grid utama, dengan algoritma terbaik di setiap pengaturan dan seberapa sering setiap algoritma menjadi yang terbaik. Tabel ini bersifat deskriptif.

### Cari Algoritma Terbaik per Pengaturan

In [ ]:
settings = config_mean_flat[
    config_mean_flat.strategy.isin(["br", "cc"]) & config_mean_flat.condition.isin(CONDITIONS_MAIN)
]
algo_by_setting = settings.pivot_table(
    index=["strategy", "condition", "tuning"], columns="algo", values="macro_f1"
)
algo_by_setting = algo_by_setting[
    [algo for algo in ALGOS if algo in algo_by_setting.columns]
].rename(columns=ALGO_NAMES)
algo_by_setting["best"] = algo_by_setting.idxmax(axis=1)
best_counts = (
    algo_by_setting["best"].value_counts().rename_axis("algorithm").to_frame("settings_where_best")
)

### Macro F1 per Pengaturan

In [ ]:
show_table(algo_by_setting)

### Jumlah Pengaturan Tempat Setiap Algoritma Terbaik

In [ ]:
show_table(best_counts)

## RQ3 Perbandingan Konfigurasi Pembanding

Setiap pasangan konfigurasi pembanding yang ditetapkan sejak awal dibandingkan dengan kedua uji dan koreksi Holm-Bonferroni atas semua pasangan. Ini adalah perbandingan konfirmatori antaralgoritma, termasuk setiap algoritma klasik terhadap IndoBERTweet.

### Bandingkan Setiap Pasangan Pembanding

In [ ]:
references = reference_models(all_runs, data)
rq3_rows = []
for name_a, name_b in combinations(references, 2):
    (pred_a, folds_a), (pred_b, folds_b) = references[name_a], references[name_b]
    rq3_rows.append(
        {
            "A": name_a,
            "B": name_b,
            **compare_models(Y, pred_a, pred_b, folds_a, folds_b, N_TRAIN, N_TEST),
        }
    )
rq3 = add_verdicts(pd.DataFrame(rq3_rows))

### Perbandingan Pembanding

In [ ]:
show_table(rq3)

## Ringkasan Pembanding

Rata-rata macro F1 setiap konfigurasi pembanding atas lima fold, dengan interval 95 persen yang memakai koreksi varians dari Nadeau dan Bengio (2003).

### Hitung Rata-Rata dan Interval

In [ ]:
reference_summary = pd.DataFrame(
    [
        {
            "model": name,
            "config": REFERENCE_CONFIGS[name],
            "macro_f1": fold_f1.mean(),
            "ci95_half_width": stats.t.ppf(0.975, 4)
            * np.sqrt((1 / 5 + N_TEST / N_TRAIN) * fold_f1.var(ddof=1)),
        }
        for name, (_, fold_f1) in references.items()
    ]
)

### Tabel Pembanding

In [ ]:
show_table(reference_summary)

### Grafik Pembanding

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(
    reference_summary.model,
    reference_summary.macro_f1,
    yerr=reference_summary.ci95_half_width,
    capsize=4,
    color=[ALGO_COLORS[model] for model in reference_summary.model],
)
ax.set_ylim(
    max(0, reference_summary.macro_f1.min() - 0.1), min(1, reference_summary.macro_f1.max() + 0.05)
)
ax.set_ylabel("Macro F1 (mean of 5 folds)")
ax.set_title("Reference configurations")
fig.savefig(FIG / "reference_macro_f1.png")
plt.show()

## F1 per Label Pembanding

F1 setiap label untuk setiap konfigurasi pembanding. Bagian ini bersifat deskriptif.

### Kumpulkan F1 per Label

In [ ]:
reference_f1 = pd.DataFrame(
    {name: per_label_and_folds(all_runs, REFERENCE_CONFIGS[name])[0] for name in references},
    index=LABELS,
).T

### Tabel F1

In [ ]:
show_table(reference_f1, decimals=3)

### Heatmap F1

In [ ]:
fig, ax = plt.subplots(figsize=(11, 0.6 * len(reference_f1) + 1.5))
sns.heatmap(
    reference_f1,
    annot=True,
    fmt=".2f",
    cmap="RdYlGn",
    vmin=0,
    vmax=1,
    cbar_kws={"label": "F1"},
    ax=ax,
)
ax.set_title("Per-label F1 of the reference configurations (mean of 5 folds)")
fig.savefig(FIG / "reference_f1_per_label.png")
plt.show()

## Perbandingan dengan Baseline

Baseline Ibrohim dan Budi (2019) memakai random forest dengan Label Powerset dan unigram kata, dan mencapai akurasi 66,12 persen pada 12 label dengan 10 fold. Akurasi pada paper tersebut adalah indeks Jaccard per dokumen, yaitu Persamaan 1 mereka. Replikasi di bawah memakai fold yang sama dengan semua model lain di sini, sehingga dapat dibandingkan di dalam penelitian ini. Angka 66,12 persen yang dipublikasikan berasal dari protokol yang berbeda, dengan sepuluh fold, tanpa deduplikasi, dan dengan fitur unigram, sehingga selisihnya hanya bersifat deskriptif.

### Baseline yang Dipublikasikan

In [ ]:
BASELINE_JACCARD_PCT = 66.12

### Kumpulkan Hasil Label Powerset

In [ ]:
lp_table = config_mean_flat[config_mean_flat.strategy == "lp"][
    ["algo", "jaccard_acc", "macro_f1", "minority_macro_f1"]
].copy()
lp_table["algo"] = lp_table.algo.map(ALGO_NAMES)
lp_table["diff_vs_published_baseline_pp"] = (
    lp_table.jaccard_acc * 100 - BASELINE_JACCARD_PCT
).round(2)

### Tabel Baseline

In [ ]:
show_table(lp_table)

## Pelanggaran Hierarki pada Prediksi

Proporsi label set prediksi yang melanggar aturan anotasi, menurut algoritma dan strategi, atas semua pengaturan grid utama.

### Hitung Hierarchy Violation Rate

In [ ]:
violations = (
    results[results.strategy.isin(["br", "cc"])]
    .groupby(["algo", "strategy"])
    .pred_hierarchy_violation_rate.agg(["mean", "min", "max"])
    .unstack("strategy")
)
violations.index = violations.index.map(ALGO_NAMES)

### Tabel Pelanggaran

In [ ]:
show_table(violations)

## Ablation Repair MLSMOTE

MLSMOTE asli dibandingkan dengan MLSMOTE dengan hierarchy repair, untuk Complement Naive Bayes dan LinearSVC dengan Binary Relevance.

### Rata-Ratakan Skor

In [ ]:
ablation = results[
    (results.strategy == "br")
    & (results.tuning == "fixed")
    & results.condition.isin(["mlsmote", "mlsmote_raw"])
    & results.algo.isin(["cnb", "svm"])
]
ablation = ablation.groupby(["algo", "condition"])[
    [
        "macro_f1",
        "minority_macro_f1",
        "jaccard_acc",
        "pred_hierarchy_violation_rate",
        "n_synthetic",
        "n_repaired",
    ]
].mean()

### Tabel Ablation

In [ ]:
show_table(ablation)

## Biaya Komputasi

Waktu training berasal dari log run dan diukur di mesin yang berbeda, sehingga bersifat deskriptif. Waktu prediksi diukur di sini pada satu mesin untuk 1.000 tweet, termasuk ekstraksi fitur atau tokenisasi. Model final adalah ensemble dari empat transformer berukuran serupa, sehingga waktu prediksinya diperkirakan sebesar empat kali satu transformer.

### Kumpulkan Biaya Training

In [ ]:
training_cost = pd.DataFrame(
    [
        {
            "model": name,
            "config": config,
            "train_time_s": all_runs[all_runs.config == config].train_time_s.mean(),
            "tune_time_s": all_runs[all_runs.config == config].tune_time_s.mean(),
            "device": ", ".join(
                sorted(all_runs[all_runs.config == config].device.dropna().astype(str).unique())
            ),
        }
        for name, config in REFERENCE_CONFIGS.items()
    ]
)

### Ukur Waktu Prediksi

Hanya dijalankan jika RUN_INFERENCE_BENCHMARK bernilai true. Waktu transformer hanya diukur jika library transformers terpasang.

In [ ]:
inference_time = None
if RUN_INFERENCE_BENCHMARK:
    inference_rows = [
        {
            "model": ALGO_NAMES[algo],
            "device": DEVICE_LABEL if algo in GPU_ALGOS else "cpu",
            "seconds_per_1000_tweets": time_classical_inference(algo, data, folds),
        }
        for algo in REFERENCE_ALGOS
    ]
    if AutoTokenizer is not None:
        transformer_texts = data.Tweet.iloc[:1000].map(normalize_for_transformer)
        transformer_seconds = time_transformer_inference(
            TRANSFORMER_DEFAULTS["model"], transformer_texts
        )
        inference_rows += [
            {
                "model": "IndoBERTweet",
                "device": DEVICE_LABEL,
                "seconds_per_1000_tweets": transformer_seconds,
            },
            {
                "model": "Final model (4 transformers, estimated)",
                "device": DEVICE_LABEL,
                "seconds_per_1000_tweets": 4 * transformer_seconds,
            },
        ]
    inference_time = pd.DataFrame(inference_rows)

### Tabel Biaya Training

In [ ]:
show_table(training_cost, decimals=2)

### Waktu Inferensi

In [ ]:
if inference_time is not None:
    show_table(inference_time)

### Grafik Inferensi

In [ ]:
if inference_time is not None:
    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.barh(inference_time.model, inference_time.seconds_per_1000_tweets, color="#2e86c1")
    ax.set_xscale("log")
    ax.set_xlabel("Seconds to label 1,000 tweets (log scale)")
    fig.savefig(FIG / "inference_time.png")
    plt.show()

# Inspeksi Fitur dan Analisis Kesalahan

Bagian ini mendaftar fitur LinearSVC yang paling berpengaruh untuk label minoritas, menampilkan false negative out-of-fold sebagai contoh, dan menguji apakah model lebih sering menandai tweet yang bukan ujaran kebencian tetapi menyebut kelompok identitas. Kata kasar disamarkan. False negative hanya berfungsi sebagai contoh, karena penyebabnya tidak di-coding oleh beberapa anotator dengan skema yang tetap.

## Pilih Konfigurasi

Konfigurasi pembanding LinearSVC yang ditetapkan sejak awal.

### Konfigurasi yang Diinspeksi

In [ ]:
INSPECT_CONFIG = REFERENCE_CONFIGS["LinearSVC"]
INSPECT_CONDITION = INSPECT_CONFIG.split("__")[2]

## Fitur Paling Berpengaruh

Sepuluh kata dengan koefisien LinearSVC terbesar untuk setiap label minoritas, dari model dengan hiperparameter tetap yang dilatih pada data latih fold 0.

### Latih pada Fold 0

In [ ]:
inspect_vectorizers = fit_features(texts_f0)
X_inspect, Y_inspect = transform_features(inspect_vectorizers, texts_f0), Y[train_rows_f0]
if INSPECT_CONDITION in RESAMPLING_CONDITIONS:
    X_inspect, Y_inspect, _ = resample(X_inspect, Y_inspect, INSPECT_CONDITION)
inspect_models = fit_binary_relevance(
    "svm", X_inspect, Y_inspect, weight=(INSPECT_CONDITION == "weight")
)

### Cari Koefisien Terbesar

In [ ]:
vocabulary = np.array(inspect_vectorizers[0].get_feature_names_out())
top_feature_lists = {}
for label in MINORITY_LABELS:
    coef = inspect_models[LABEL_INDEX[label]].coef_.ravel()
    top = np.argsort(coef)[::-1][:10]
    top_feature_lists[label] = [
        f"{mask_abusive_words(vocabulary[feature], abusive_lexicon)} ({coef[feature]:.2f})"
        for feature in top
    ]
top_features = pd.DataFrame(top_feature_lists)

### Fitur Teratas

In [ ]:
top_features

## False Negative

False negative out-of-fold dari pembanding LinearSVC untuk label minoritas. Jumlahnya lengkap, dan yang ditampilkan adalah sepuluh baris pertama dari sampel acak contoh.

### Bangun Prediksi Out-of-Fold

In [ ]:
row_of_id = pd.Series(np.arange(len(data)), index=data["id"])
oof_scores = np.zeros_like(Y, dtype=float)
for fold in range(5):
    test_ids, fold_scores = load_fold_scores(INSPECT_CONFIG, fold)
    oof_scores[row_of_id[test_ids].to_numpy()] = fold_scores
oof_pred = (oof_scores > 0).astype(int)

### Hitung Kesalahan

In [ ]:
error_counts = pd.DataFrame(
    {
        label: {
            "positives": int(Y[:, LABEL_INDEX[label]].sum()),
            "false_negatives": int(
                ((Y[:, LABEL_INDEX[label]] == 1) & (oof_pred[:, LABEL_INDEX[label]] == 0)).sum()
            ),
            "false_positives": int(
                ((Y[:, LABEL_INDEX[label]] == 0) & (oof_pred[:, LABEL_INDEX[label]] == 1)).sum()
            ),
        }
        for label in MINORITY_LABELS
    }
).T

### Ambil Sampel False Negative

Paling banyak sepuluh false negative acak per label minoritas.

In [ ]:
fn_rows, rng_fn = [], np.random.default_rng(RANDOM_STATE)
for label in MINORITY_LABELS:
    label_index = LABEL_INDEX[label]
    false_negatives = np.where((Y[:, label_index] == 1) & (oof_pred[:, label_index] == 0))[0]
    for row in rng_fn.choice(false_negatives, size=min(10, len(false_negatives)), replace=False):
        fn_rows.append(
            {
                "label": label,
                "id": int(data.id.iloc[row]),
                "score": round(oof_scores[row, label_index], 3),
                "true_labels": ", ".join(name for name, value in zip(LABELS, Y[row]) if value),
                "predicted_labels": ", ".join(
                    name for name, value in zip(LABELS, oof_pred[row]) if value
                )
                or "-",
                "tweet_masked": mask_abusive_words(data.Tweet.iloc[row], abusive_lexicon),
            }
        )
fn_examples = pd.DataFrame(fn_rows)

### Jumlah Kesalahan

In [ ]:
error_counts

### Contoh

Hanya sepuluh baris pertama yang ditampilkan.

In [ ]:
show_table(fn_examples, head=10)

## Kategori yang Terlewat

Jika sebuah kategori terlewat, tabel ini menunjukkan kategori mana yang diprediksi sebagai gantinya dan seberapa sering HS sendiri terlewat.

### Hitung Prediksi untuk Kategori yang Terlewat

In [ ]:
missed_rows = {}
for label in ["HS_Gender", "HS_Physical", "HS_Race", "HS_Religion"]:
    missed = (Y[:, LABEL_INDEX[label]] == 1) & (oof_pred[:, LABEL_INDEX[label]] == 0)
    missed_rows[label] = {
        category: int(oof_pred[missed, LABEL_INDEX[category]].sum()) for category in CATEGORY_LABELS
    } | {
        "not_predicted_HS": int((oof_pred[missed, HS_INDEX] == 0).sum()),
        "total_FN": int(missed.sum()),
    }
category_confusion = pd.DataFrame(missed_rows).T

### Kategori yang Diprediksi sebagai Gantinya

In [ ]:
category_confusion

## Identity Term

Identity term di bawah ditulis sebelum uji bias dijalankan. Istilah ini menyebut kelompok agama, etnis, dan gender dengan kata netral, tanpa makian, dalam bentuk teks yang sudah dibersihkan dan dinormalisasi. Tim perlu meninjau daftar ini sebelum run pertama dan tidak boleh mengubahnya setelah melihat hasilnya.

### Kelompok Istilah

In [ ]:
IDENTITY_TERMS = {
    "religion": [
        "islam",
        "muslim",
        "muslimah",
        "kristen",
        "katolik",
        "protestan",
        "nasrani",
        "hindu",
        "buddha",
        "budha",
        "konghucu",
        "yahudi",
    ],
    "ethnicity": [
        "cina",
        "china",
        "tionghoa",
        "arab",
        "jawa",
        "sunda",
        "batak",
        "minang",
        "padang",
        "bugis",
        "madura",
        "papua",
        "betawi",
        "ambon",
        "india",
        "pribumi",
    ],
    "gender": [
        "perempuan",
        "wanita",
        "cewek",
        "pria",
        "cowok",
        "gay",
        "lesbian",
        "lesbi",
        "lgbt",
        "transgender",
    ],
}

### Daftar Istilah

In [ ]:
pd.Series(
    {group: ", ".join(terms) for group, terms in IDENTITY_TERMS.items()}, name="terms"
).to_frame()

## Bias Identity Term

Di antara tweet yang bukan ujaran kebencian, false positive rate label HS dibandingkan antara tweet yang mengandung identity term dan tweet yang tidak, mengikuti evaluasi unintended bias dari Dixon et al. (2018). Fisher's exact test memeriksa selisihnya, dengan koreksi Holm-Bonferroni atas semua baris. Rate yang lebih tinggi pada tweet dengan identity term berarti model bereaksi terhadap nama kelompok itu sendiri.

### Cari Tweet dengan Identity Term

Satu mask untuk setiap kelompok istilah, satu untuk istilah apa pun, dan mask untuk tweet yang bukan ujaran kebencian.

In [ ]:
tokens = data.text_clean.str.split().map(set)
group_masks = {
    group: tokens.map(
        lambda tweet_tokens, terms=frozenset(terms): bool(tweet_tokens & terms)
    ).to_numpy()
    for group, terms in IDENTITY_TERMS.items()
}
group_masks["any identity term"] = np.logical_or.reduce(list(group_masks.values()))
not_hate = Y[:, HS_INDEX] == 0

### Jalankan Fisher's Exact Test

Satu uji untuk setiap konfigurasi pembanding dan kelompok istilah.

In [ ]:
bias_rows = []
for (name, (pred, _)), (group, mask) in product(references.items(), group_masks.items()):
    false_positive = (pred[:, HS_INDEX] == 1) & not_hate
    with_term, without_term = not_hate & mask, not_hate & ~mask
    fp_with, fp_without = int(false_positive[with_term].sum()), int(
        false_positive[without_term].sum()
    )
    odds_ratio, p_value = stats.fisher_exact(
        [
            [fp_with, int(with_term.sum()) - fp_with],
            [fp_without, int(without_term.sum()) - fp_without],
        ]
    )
    bias_rows.append(
        {
            "model": name,
            "group": group,
            "non_hate_tweets_with_term": int(with_term.sum()),
            "fpr_with_term": fp_with / max(int(with_term.sum()), 1),
            "fpr_without_term": fp_without / max(int(without_term.sum()), 1),
            "odds_ratio": odds_ratio,
            "p_fisher": p_value,
        }
    )
identity_bias = add_holm_columns(pd.DataFrame(bias_rows), ["p_fisher"])

### Tabel Bias

In [ ]:
show_table(identity_bias)

### Grafik Bias

False positive rate untuk identity term apa pun.

In [ ]:
any_term = identity_bias[identity_bias.group == "any identity term"]
positions = np.arange(len(any_term))
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.bar(positions - 0.2, any_term.fpr_with_term, 0.4, label="with identity term", color="#c0392b")
ax.bar(
    positions + 0.2, any_term.fpr_without_term, 0.4, label="without identity term", color="#7f8c8d"
)
ax.set_xticks(positions)
ax.set_xticklabels(any_term.model)
ax.set_ylabel("False positive rate of HS")
ax.set_title("Non-hateful tweets flagged as hate speech")
ax.legend()
fig.savefig(FIG / "identity_bias.png")
plt.show()

# Eksperimen Lanjutan

Eksperimen ini menjawab pertanyaan yang lazim diajukan reviewer dan menguji perbaikan dari literatur. Semua run masuk ke results/experiments_ext.csv, dengan decision score di results/preds_ext, sehingga analisis utama tidak berubah.

Eksperimen lanjutan menanyakan apakah post-processing dengan hierarki anotasi memperbaiki prediksi tanpa training ulang, apakah oversampling acak dengan ML-ROS juga gagal seperti MLSMOTE, apakah n-gram karakter membantu model klasik, mengapa oversampling gagal, dan apakah transformer dapat ditingkatkan dengan pemilihan epoch, model lain, dan ensemble. Eksperimen ini diakhiri dengan pemilihan strategi threshold dan model final, serta perbandingan model final dengan konfigurasi pembanding.

Setiap pemilihan hiperparameter, threshold, epoch, threshold ensemble, dan konfigurasi final hanya memakai inner validation dari outer training fold. Konfigurasi transformer dikembangkan bertahap setelah hasil sebelumnya dilihat, sehingga model final bersifat eksploratif, dan setiap konfigurasi yang pernah dicoba dilaporkan.

## Pengaturan Transformer

Konfigurasi dan ensemble transformer dalam protokol, dengan key berupa nama konfigurasi. Ensemble merata-ratakan probabilitas anggotanya dan melakukan tuning threshold pada gabungan data inner validation.

### Konfigurasi Transformer

Setiap konfigurasi menimpa sebagian pengaturan default protokol.

In [ ]:
TRANSFORMER_CONFIGS = {
    "indobertweet_v2": {"model": "indolem/indobertweet-base-uncased"},
    "indobert_p1_v2": {"model": "indobenchmark/indobert-base-p1"},
    "indobertweet_v2_e10": {"model": "indolem/indobertweet-base-uncased", "epochs": 10},
    "indobertweet_v2_s1": {"model": "indolem/indobertweet-base-uncased", "seed": 1},
    "indobertweet_v2_s2": {"model": "indolem/indobertweet-base-uncased", "seed": 2},
    "indobertweet_v2_e10_s1": {
        "model": "indolem/indobertweet-base-uncased",
        "epochs": 10,
        "seed": 1,
    },
    "indobertweet_v2_e10_s2": {
        "model": "indolem/indobertweet-base-uncased",
        "epochs": 10,
        "seed": 2,
    },
    "indobert_p1_v2_e10": {"model": "indobenchmark/indobert-base-p1", "epochs": 10},
}

### Ensemble

Konfigurasi anggota dari setiap ensemble.

In [ ]:
TRANSFORMER_ENSEMBLES = {
    "indobertweet+indobert_p1": ["indobertweet_v2", "indobert_p1_v2"],
    "indobertweet_x3": ["indobertweet_v2", "indobertweet_v2_s1", "indobertweet_v2_s2"],
    "indobertweet_x3+indobert_p1": [
        "indobertweet_v2",
        "indobertweet_v2_s1",
        "indobertweet_v2_s2",
        "indobert_p1_v2",
    ],
    "e10_indobertweet+indobert_p1": ["indobertweet_v2_e10", "indobert_p1_v2_e10"],
    "e10_indobertweet_x3": [
        "indobertweet_v2_e10",
        "indobertweet_v2_e10_s1",
        "indobertweet_v2_e10_s2",
    ],
    "e10_indobertweet_x3+indobert_p1": [
        "indobertweet_v2_e10",
        "indobertweet_v2_e10_s1",
        "indobertweet_v2_e10_s2",
        "indobert_p1_v2_e10",
    ],
}

### Tabel Konfigurasi

In [ ]:
pd.DataFrame(TRANSFORMER_CONFIGS).T.fillna("default")

### Anggota Ensemble

In [ ]:
pd.Series(
    {name: " + ".join(members) for name, members in TRANSFORMER_ENSEMBLES.items()}, name="members"
).to_frame()

## Terapkan Post-Processing

Menerapkan aturan dari bagian Post-Processing dengan Hierarki pada setiap konfigurasi Binary Relevance, Classifier Chains, dan transformer yang lengkap. Konfigurasi pembanding juga diuji terhadap prediksinya sendiri tanpa post-processing.

### Post-Processing Setiap Konfigurasi

In [ ]:
all_runs = load_all_runs()
all_runs["tuning"] = tuning_label(all_runs)
n_folds = all_runs[all_runs.strategy.isin(["br", "cc", "mo"])].groupby("config").fold.nunique()
postprocessing_rows, reference_test_rows = [], []
for config in n_folds[n_folds == 5].index:
    oof, fold_metrics = postprocess_config(config, data)
    postprocessing_rows.append(
        postprocessing_row(config, all_runs[all_runs.config == config].iloc[0], fold_metrics)
    )
    if config in REFERENCE_CONFIGS.values():
        folds_original = np.array(
            [fold_result["macro_f1"] for fold_result in fold_metrics["original"]]
        )
        folds_topdown = np.array(
            [fold_result["macro_f1"] for fold_result in fold_metrics["topdown"]]
        )
        reference_test_rows.append(
            {
                "A": f"{config} original",
                "B": f"{config} topdown",
                **compare_models(
                    Y,
                    oof["original"],
                    oof["topdown"],
                    folds_original,
                    folds_topdown,
                    N_TRAIN,
                    N_TEST,
                ),
            }
        )
postprocessing = pd.DataFrame(postprocessing_rows)

### Ringkas Hasil

Berapa banyak konfigurasi yang membaik, rata-rata perubahan menurut algoritma dan strategi, dan verdict dari uji pada konfigurasi pembanding.

In [ ]:
postprocessing_counts = pd.DataFrame(
    [
        {
            "configurations": len(postprocessing),
            "macro_f1_up_topdown": int((postprocessing.delta_macro__topdown > 0).sum()),
            "jaccard_up_topdown": int((postprocessing.delta_jacc__topdown > 0).sum()),
        }
    ]
)
postprocessing_by_algo = postprocessing.groupby(["algo", "strategy"])[
    ["delta_macro__topdown", "delta_jacc__topdown", "delta_macro__bottomup", "delta_jacc__bottomup"]
].mean()
postprocessing_tests = (
    add_verdicts(pd.DataFrame(reference_test_rows)) if reference_test_rows else None
)

### Konfigurasi yang Membaik

In [ ]:
postprocessing_counts

### Rata-Rata Perubahan menurut Algoritma dan Strategi

In [ ]:
show_table(postprocessing_by_algo)

### Per Konfigurasi

Hanya sepuluh baris pertama yang ditampilkan, diurutkan berdasarkan macro F1 tanpa post-processing.

In [ ]:
show_table(postprocessing.sort_values("macro_f1__original", ascending=False), head=10)

### Uji pada Konfigurasi Pembanding

In [ ]:
if postprocessing_tests is not None:
    show_table(postprocessing_tests)

## Grid ML-ROS dan Fitur Karakter

ML-ROS menggandakan instance minoritas hingga 20 persen dari data latih, mendekati proporsi yang ditambahkan MLSMOTE. Jika ML-ROS dan MLSMOTE sama-sama tidak membantu, kesimpulan bahwa oversampling tidak cocok untuk fitur TF-IDF pada data ini menjadi lebih kuat dan tidak bergantung pada implementasi MLSMOTE. Fitur karakter serta gabungan fitur kata dan karakter diuji karena Ibrohim dan Budi (2019) melaporkan fitur karakter termasuk yang terbaik pada data ini. ML-ROS dijalankan dengan Complement Naive Bayes dan LinearSVC, dan fitur karakter dengan LinearSVC, sesuai pengaturan di konfigurasi.

### Grid Lanjutan

In [ ]:
extended_grid = build_grid(
    [algo for algo in MLROS_ALGOS if algo in CLASSICAL_ALGOS],
    ["br", "cc"],
    ["mlros"],
    tuning=("fixed", "tuned"),
)
extended_grid += build_grid(
    [algo for algo in CHAR_FEATURE_ALGOS if algo in CLASSICAL_ALGOS],
    ["br", "cc"],
    ["none"],
    tuning=("tuned",),
    features=("char", "wordchar"),
)

### Jalankan Grid Lanjutan

Run yang sudah ada di experiments_ext.csv dilewati. Log run kemudian dimuat ulang bersama run yang baru.

In [ ]:
if RUN_EXTENDED_GRID:
    run_grid(
        ROOT,
        extended_grid,
        n_jobs_cpu=N_JOBS_CPU,
        csv_name=EXTENDED_CSV,
        preds_subdir="preds_ext",
        verbose=False,
    )
extended = load_extended()
all_runs = load_all_runs()
complete_configs = {
    config for config, n_done in all_runs.groupby("config").fold.nunique().items() if n_done == 5
}
EXTENDED_ALGOS = [algo for algo in ["cnb", "svm", "xgb", "mlp"] if algo in set(extended.algo)]

### Run di Log Lanjutan

In [ ]:
extended.groupby("algo").agg(configurations=("config", "nunique"), runs=("config", "size"))

## Perbandingan ML-ROS dan Fitur

ML-ROS dibandingkan dengan penanganan ketidakseimbangan lain dalam mode tuning yang sama, dan fitur karakter dibandingkan dengan fitur kata, memakai kedua uji dan koreksi Holm-Bonferroni di dalam setiap family.

### Pasangan Perbandingan

In [ ]:
mlros_pairs = [
    {
        "family": "ML-ROS",
        "algo": ALGO_NAMES[algo],
        "strategy": strategy.upper(),
        "config_a": config_id(algo, strategy, other, tuning),
        "config_b": config_id(algo, strategy, "mlros", tuning),
    }
    for algo, strategy, tuning, other in product(
        EXTENDED_ALGOS, ["br", "cc"], ["fixed", "tuned"], ["none", "mlsmote", "weight"]
    )
]
feature_pairs = [
    {
        "family": "features",
        "algo": ALGO_NAMES[algo],
        "strategy": strategy.upper(),
        "config_a": config_id(algo, strategy, "none", "tuned"),
        "config_b": config_id(algo, strategy, "none", "tuned", features),
    }
    for algo, strategy, features in product(CHAR_FEATURE_ALGOS, ["br", "cc"], ["char", "wordchar"])
]

### Jalankan Perbandingan

Pasangan yang konfigurasinya belum lengkap dilewati.

In [ ]:
extended_comparisons = add_verdicts(
    compare_configs(mlros_pairs + feature_pairs, all_runs, data, N_TRAIN, N_TEST), by="family"
)

### Verdict per Family

In [ ]:
pd.crosstab(extended_comparisons.family, extended_comparisons.verdict)

### Semua Perbandingan

In [ ]:
show_table(extended_comparisons)

## Precision dan Recall Label Minoritas

Tabel ini menguji satu penjelasan mengapa oversampling tidak membantu, yaitu recall label minoritas naik sementara precision-nya turun lebih jauh. Nilainya adalah rata-rata atas lima fold untuk Binary Relevance dengan fitur kata, di setiap mode tuning.

### Kumpulkan Precision dan Recall per Pengaturan

In [ ]:
minority_pr_rows = []
for algo, tuning, condition in product(
    EXTENDED_ALGOS, ["fixed", "thr", "tuned"], ["none", "weight", "mlsmote", "mlros"]
):
    config = config_id(algo, "br", condition, tuning)
    if config not in complete_configs:
        continue
    runs = all_runs[all_runs.config == config]
    row = {"algo": ALGO_NAMES[algo], "tuning": tuning, "condition": condition}
    row |= {f"P_{label}": runs[f"p__{label}"].mean() for label in MINORITY_LABELS}
    row |= {f"R_{label}": runs[f"r__{label}"].mean() for label in MINORITY_LABELS}
    row["P_mean"] = np.mean([row[f"P_{label}"] for label in MINORITY_LABELS])
    row["R_mean"] = np.mean([row[f"R_{label}"] for label in MINORITY_LABELS])
    minority_pr_rows.append(row)
minority_pr = pd.DataFrame(minority_pr_rows)

### Tabel Precision dan Recall

In [ ]:
show_table(minority_pr, decimals=3)

## Protokol Transformer

Setiap konfigurasi di TRANSFORMER_CONFIGS dilatih dengan protokol pada lima fold, dan ensemble hanya memakai probabilitas yang tersimpan. Konfigurasi final dipilih berdasarkan macro F1 inner validation, bukan skor uji, dan skor ujinya dilaporkan apa adanya. Preprocessing-nya mengikuti IndoBERTweet (Koto et al., 2021).

### Latih dan Muat

Fold yang sudah tersimpan dilewati.

In [ ]:
if RUN_TRANSFORMERS:
    run_transformer_configs(ROOT, TRANSFORMER_CONFIGS)
    for ensemble_name, members in TRANSFORMER_ENSEMBLES.items():
        ensemble_transformers(ROOT, members, ensemble_name)
extended = load_extended()
transformer_runs = (
    extended[extended.algo.isin(["tf", "ens"])].copy() if len(extended) else pd.DataFrame()
)
HAS_TRANSFORMERS = len(transformer_runs) > 0

### Run Transformer Tersimpan

In [ ]:
transformer_runs.groupby("algo").agg(configurations=("config", "nunique"), runs=("config", "size"))

## Tabel Ringkasan Transformer

Setiap konfigurasi dan ensemble transformer yang memiliki kelima fold, serta konfigurasi dengan macro F1 inner validation terbaik pada threshold hasil tuning.

### Ringkas Konfigurasi

In [ ]:
if HAS_TRANSFORMERS:
    n_folds = transformer_runs.groupby("config").fold.nunique()
    transformer_runs = transformer_runs[transformer_runs.config.isin(n_folds[n_folds == 5].index)]
    columns = [
        "macro_f1",
        "micro_f1",
        "minority_macro_f1",
        "jaccard_acc",
        "inner_macro_f1",
        "train_time_s",
    ]
    transformer_table = transformer_runs.groupby("config")[columns].agg(["mean", "std"])
    transformer_table.columns = [
        f"{metric}_{statistic}" for metric, statistic in transformer_table.columns
    ]
    if "best_epoch" in transformer_runs:
        transformer_table["best_epochs"] = transformer_runs.groupby("config").best_epoch.apply(
            lambda epochs: "/".join(map(str, epochs.astype("Int64")))
        )
    transformer_table = transformer_table.sort_values("macro_f1_mean", ascending=False)
    tuned_runs = transformer_runs[transformer_runs.config.str.endswith("__tuned")]
    picked = tuned_runs[
        tuned_runs.config == tuned_runs.groupby("config").inner_macro_f1.mean().idxmax()
    ]
    best_by_inner = pd.DataFrame(
        [
            {
                "best_by_inner_validation": picked.config.iloc[0],
                "inner_macro_f1": picked.inner_macro_f1.mean(),
                "test_macro_f1": picked.macro_f1.mean(),
                "test_macro_f1_std": picked.macro_f1.std(),
            }
        ]
    )

### Semua Konfigurasi

In [ ]:
if HAS_TRANSFORMERS:
    show_table(transformer_table)

### Terbaik menurut Inner Validation

In [ ]:
if HAS_TRANSFORMERS:
    show_table(best_by_inner)

## Strategi Threshold dan Model Final

Strategi dan estimasi inner validation-nya dijelaskan pada bagian Strategi Threshold. Konfigurasi dan strategi dengan estimasi inner validation tertinggi menjadi model final, tanpa training ulang.

### Evaluasi Strategi Threshold

Setiap strategi pada setiap fold dari setiap konfigurasi dan ensemble transformer, diikuti pemilihan model final.

In [ ]:
if HAS_TRANSFORMERS:
    Y_BY_ID = data.set_index("id")[LABELS]
    base_configs = sorted(
        {config.removesuffix("__tuned") for config in transformer_runs.config.unique()}
    )
    threshold_rows, threshold_store = [], {}
    for base, fold in product(base_configs, range(5)):
        evaluation = evaluate_threshold_strategies(
            Y_BY_ID, RES / "preds_ext" / f"{base}__f{fold}.npz", seed=fold
        )
        for strategy, values in evaluation.items():
            threshold_rows.append(
                {
                    "config": base,
                    "fold": fold,
                    "strategy": strategy,
                    "inner_2fold": values["inner_2fold"],
                    "test_macro_f1": values["test_macro_f1"],
                }
            )
            threshold_store[(base, strategy, fold)] = values
    threshold_results = pd.DataFrame(threshold_rows)
    inner_estimate = threshold_results.groupby(["config", "strategy"]).inner_2fold.mean()
    FINAL_CONFIG, FINAL_STRATEGY = inner_estimate.idxmax()

### Prediksi dengan Model Final

Prediksi out-of-fold dari model final dan skornya pada setiap fold.

In [ ]:
if HAS_TRANSFORMERS:
    row_of_id = pd.Series(np.arange(len(data)), index=data["id"])
    final_pred, final_rows = np.zeros_like(Y), []
    for fold in range(5):
        values = threshold_store[(FINAL_CONFIG, FINAL_STRATEGY, fold)]
        rows = row_of_id[values["test_ids"]].to_numpy()
        fold_pred = (values["test_prob"] > values["thresholds"][None, :]).astype(int)
        final_pred[rows] = fold_pred
        final_rows.append({"fold": fold, **evaluate_predictions(Y[rows], fold_pred)})
    final_model_folds = pd.DataFrame(final_rows)
    final_model_summary = pd.DataFrame(
        [
            {
                "final_config": FINAL_CONFIG,
                "threshold_strategy": FINAL_STRATEGY,
                "inner_2fold": inner_estimate.max(),
                "macro_f1": final_model_folds.macro_f1.mean(),
                "macro_f1_std": final_model_folds.macro_f1.std(),
                "minority_macro_f1": final_model_folds.minority_macro_f1.mean(),
                "micro_f1": final_model_folds.micro_f1.mean(),
                "jaccard_acc": final_model_folds.jaccard_acc.mean(),
                "subset_acc": final_model_folds.subset_acc.mean(),
                "hamming_accuracy": 1 - final_model_folds.hamming_loss.mean(),
            }
        ]
    )

### Perbandingan Strategi Threshold

Estimasi inner validation dan macro F1 uji dari setiap strategi, sebagai rata-rata atas lima fold.

In [ ]:
if HAS_TRANSFORMERS:
    show_table(
        threshold_results.groupby(["config", "strategy"])[["inner_2fold", "test_macro_f1"]]
        .mean()
        .unstack("strategy")
    )

### Model Final

In [ ]:
if HAS_TRANSFORMERS:
    show_table(final_model_summary)

### Model Final per Fold

In [ ]:
if HAS_TRANSFORMERS:
    show_table(
        final_model_folds[
            [
                "fold",
                "macro_f1",
                "micro_f1",
                "minority_macro_f1",
                "jaccard_acc",
                "subset_acc",
                "hamming_loss",
            ]
        ]
    )

## Model Final terhadap Konfigurasi Pembanding

F1 setiap label untuk model final dan setiap konfigurasi pembanding, diikuti uji berpasangan model final terhadap setiap pembanding dengan koreksi Holm-Bonferroni.

### Bandingkan dengan Konfigurasi Pembanding

In [ ]:
if HAS_TRANSFORMERS:
    final_runs = load_all_runs()
    final_references = reference_models(final_runs, data)
    final_per_label = pd.DataFrame(
        {
            name: per_label_and_folds(final_runs, REFERENCE_CONFIGS[name])[0]
            for name in final_references
        },
        index=LABELS,
    )
    final_per_label["Final model"] = [final_model_folds[f"f1__{label}"].mean() for label in LABELS]
    final_per_label.loc["macro-F1"] = final_per_label.mean()
    folds_final = final_model_folds.sort_values("fold").macro_f1.to_numpy()
    final_comparison = add_verdicts(
        pd.DataFrame(
            [
                {
                    "A": name,
                    "B": "Final model",
                    **compare_models(Y, pred, final_pred, folds, folds_final, N_TRAIN, N_TEST),
                }
                for name, (pred, folds) in final_references.items()
            ]
        )
    )

### F1 per Label Model Final

In [ ]:
if HAS_TRANSFORMERS:
    show_table(final_per_label, decimals=3)

### Uji terhadap Konfigurasi Pembanding

In [ ]:
if HAS_TRANSFORMERS:
    show_table(final_comparison)

# Evaluasi Lintas Korpus

Model dilatih pada seluruh data Ibrohim dan Budi (2019) dan diuji pada rilis terbaru IndoToxic2024, yaitu IndoDiscourse versi 2 dari Susanto et al. (2024). Korpus ini didistribusikan dengan lisensi Apache 2.0 dan diunduh dari dataset HuggingFace Exqrch/IndoDiscourse.

Pemetaan label adalah sebuah asumsi. HS dipetakan ke toxicity, yang dipakai paper IndoToxic2024 sebagai task ujaran kebencian, dan ke identity_attack sebagai perbandingan yang lebih sempit. Toxicity adalah konstruk yang lebih luas daripada ujaran kebencian, sehingga ini adalah transfer ke konstruk yang berkaitan, dan penurunan F1 mencampurkan kemampuan generalisasi dengan perbedaan definisi label, platform, tahun, dan prevalensi. F1 dari classifier yang selalu memprediksi positif menunjukkan F1 yang dapat dicapai hanya dari prevalensi. Abusive yang dipetakan ke profanity_obscenity memiliki prevalensi sekitar 1 persen dan definisi yang berbeda, sehingga ditampilkan terpisah dan tidak diinterpretasikan.

Modelnya adalah keempat konfigurasi pembanding klasik, dengan hiperparameter dan threshold dari 15 persen inner validation, ditambah transformer, model final, dan ensemble pembandingnya dengan threshold 0,5.

## Unduh dan Beri Skor

Mengunduh korpus jika belum ada dan memberi skor setiap model pada korpus tersebut.

### Unduh Korpus

Membutuhkan akses internet pada run pertama. Evaluasi dijalankan jika RUN_CROSS_CORPUS bernilai true, atau jika bernilai None dan file korpus sudah ada.

In [ ]:
if not CROSS_CORPUS_FILE.exists():
    try:
        from huggingface_hub import hf_hub_download

        CROSS_CORPUS_FILE.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(
            hf_hub_download(
                "Exqrch/IndoDiscourse",
                "indotoxic2024_annotated_data_v2_final.csv",
                repo_type="dataset",
            ),
            CROSS_CORPUS_FILE,
        )
    except Exception as error:
        print("Could not download the 2024 corpus:", repr(error))
if RUN_CROSS_CORPUS is None:
    RUN_CROSS_CORPUS = CROSS_CORPUS_FILE.exists()

### Beri Skor Setiap Model

Melatih keempat pembanding klasik pada seluruh data, yang memakan waktu beberapa menit, dan memakai ulang probabilitas transformer yang tersimpan.

In [ ]:
if RUN_CROSS_CORPUS:
    corpus_2024, cross_corpus, oov_summary, oov_top = run_cross_corpus(
        ROOT, run_transformers=RUN_TRANSFORMERS
    )
    corpus_summary = pd.DataFrame(
        [
            {
                "texts": len(corpus_2024),
                "toxicity_rate": corpus_2024.toxicity.mean(),
                "identity_attack_rate": corpus_2024.identity_attack.mean(),
                "profanity_rate": corpus_2024.profanity_obscenity.mean(),
            }
        ]
    ).round(3)

### Ringkasan Korpus

Korpus setelah anotasinya diagregasi.

In [ ]:
if RUN_CROSS_CORPUS:
    display(corpus_summary)

## Perbandingan In-Domain

Menambahkan F1 in-domain untuk label yang sama dari cross-validation pada Ibrohim dan Budi, serta penurunannya pada korpus 2024.

### Tambahkan F1 In-Domain

In [ ]:
if RUN_CROSS_CORPUS:
    cross_runs = load_all_runs()
    cross_corpus["f1_in_domain_cv"] = [
        in_domain_f1(cross_runs, model, label)
        for model, label in zip(cross_corpus.model, cross_corpus.source_label)
    ]
    cross_corpus["f1_drop"] = cross_corpus.f1_in_domain_cv - cross_corpus.f1_pos
    hate_mappings = cross_corpus[cross_corpus.target_label != "profanity_obscenity"]
    profanity_mapping = cross_corpus[cross_corpus.target_label == "profanity_obscenity"]

### Pemetaan Ujaran Kebencian

In [ ]:
if RUN_CROSS_CORPUS:
    show_table(hate_mappings)

### Pemetaan Kata Kasar

Pemetaan ini tidak diinterpretasikan, sehingga hanya lima baris pertama yang ditampilkan.

In [ ]:
if RUN_CROSS_CORPUS:
    show_table(profanity_mapping, head=5)

### Vocabulary Overlap

In [ ]:
if RUN_CROSS_CORPUS:
    display(pd.DataFrame([oov_summary]).round(3))

### Kata Out-of-Vocabulary yang Paling Sering Muncul

Hanya lima belas baris pertama yang ditampilkan.

In [ ]:
if RUN_CROSS_CORPUS:
    show_table(oov_top, head=15)

## Grafik Lintas Korpus

F1 kelas positif pada IndoToxic2024 dengan interval bootstrap 95 persennya, untuk kedua pemetaan ujaran kebencian. Garis putus-putus adalah F1 dari classifier yang selalu memprediksi positif.

### F1 dengan Interval Bootstrap

In [ ]:
if RUN_CROSS_CORPUS:
    targets = ["toxicity", "identity_attack"]
    fig, axes = plt.subplots(1, len(targets), figsize=(14, 6), sharey=True)
    for ax, target in zip(axes, targets):
        subset = hate_mappings[hate_mappings.target_label == target]
        error = [subset.f1_pos - subset.f1_ci95_low, subset.f1_ci95_high - subset.f1_pos]
        ax.barh(subset.model, subset.f1_pos, xerr=error, capsize=3, color="#2e86c1")
        ax.axvline(subset.f1_always_positive.iloc[0], color="black", ls="--", lw=1)
        ax.set_xlabel(f"F1 of {target} on IndoToxic2024")
        ax.set_title(target)
    fig.tight_layout()
    fig.savefig(FIG / "crosscorpus_f1.png")
    plt.show()